# AMML Ranking Competition 2026 — финальное решение

**Итог:** Public NDCG@10 `0.964651`, Private NDCG@10 `0.967919`, **10-е место**.  
**Окружение оригинальных запусков:** Kaggle Linux (x86_64), Python 3.13, **2 × Tesla T4**, PyTorch `2.11.0+cu130`; версии библиотек в `requirements.txt`.  
**Вход:** только оригинальные CSV соревнования и публичные **предобученные веса** Qwen/Jina (не внешние датасеты).  
**Выход финального этапа:** `/kaggle/working/amml_repro_cache/submission.csv` (`id,prediction`, 13 507 строк).

### Как воспроизводить

Из-за суммарной длительности обучения решение разбито **на последовательные стадии одного и того же ноутбука**. На каждом коммите меняйте `STAGE` и запускайте `Save Version → Save & Run All`. Между коммитами подключайте через **Add Input → Notebook Output** **только результат непосредственно предыдущей стадии**, содержащий `amml_repro_cache/`. Исходные данные соревнования также должны оставаться подключёнными. Скрипты создаются **самим ноутбуком**; дополнительные `.py`-файлы в GitHub не требуются. После каждой стадии Kaggle сохраняет чекпоинты внутри `amml_repro_cache/`. Не подключайте старые outputs из реальных соревнований: воспроизведение рассчитано на обучение с нуля.

1. `check_data` — проверка исходных CSV (CPU).
2. `qwen4b` — Qwen3-Reranker-4B, QLoRA (GPU).
3. `qwen8b` — Qwen3-Reranker-8B, QLoRA (GPU).
4. `qwen_v6_4b` — новый инференс *обученной* 4B (GPU, без дополнительной оптимизации).
5. `qwen_v6_8b` — новый инференс *обученной* 8B (GPU).
6. `qwen_combine` — зафиксированное 50/50 v5 и 75/25 v5/4B-view (CPU).
7. `jina_fast` — обучение Jina-reranker (GPU).
8. `jina_a` — продолженное обучение Jina A (GPU).
9. `jina_b` — обучение Jina B от исходных весов (GPU).
10. `jina_combine` — сборка исходной Jina и вариантов A/B (CPU).
11. `view_body` — Jina без заголовка, 384 токена (GPU).
12. `view_long` — Jina B с расширенным контекстом, 512 токенов (GPU).
13. `final` — **только** финальный Rank Mean и `submission.csv` (CPU).

Модели QA и старые baseline-эксперименты не обучаются: они не входят в итоговый скор. Параметры обучения и фиксированные версии весов взяты из сохранённых `run.json`; обучающиеся стадии нельзя пропускать. Для GPU-стадий требуется **T4 × 2** (используется по одному GPU на стадию), Internet ON для загрузки предобученных весов. CPU-стадии могут запускаться без ускорителя. Результаты GPU могут не совпасть побитово из-за CUDA и порядка операций. Общее время существенно больше одного Kaggle-коммита.

**Честное ограничение:** этот объединённый notebook прошёл синтаксические/локальные тесты, но ещё **не проходил полный независимый повтор всех GPU-стадий**. Полученный Private Score относится к исходному успешному пайплайну. После независимого прогона сравните новый `submission.csv` с исходным по `id` и порядку внутри `query_id`; не утверждайте побитовую идентичность заранее.


In [ ]:
# CHANGE ONLY THIS VALUE for the next separate Kaggle Commit.
STAGE = "check_data"  # check_data, qwen4b, qwen8b, qwen_v6_4b, qwen_v6_8b,
                      # qwen_combine, jina_fast, jina_a, jina_b, jina_combine,
                      # view_body, view_long, final

INSTALL_DEPENDENCIES = True

import os, sys, json, shutil, time, subprocess, importlib.util
from pathlib import Path
INPUT = Path('/kaggle/input')
CACHE = Path('/kaggle/working/amml_repro_cache')
CACHE.mkdir(parents=True, exist_ok=True)

STAGES = ['check_data','qwen4b','qwen8b','qwen_v6_4b','qwen_v6_8b',
          'qwen_combine','jina_fast','jina_a','jina_b','jina_combine',
          'view_body','view_long','final']
assert STAGE in STAGES, STAGE

# Use one previous saved Notebook Output (NOT all the older outputs).
previous = sorted(INPUT.rglob('amml_repro_cache/_completed.json'))
assert len(previous) <= 1, f'Attach exactly one previous Notebook Output: {previous}'
if previous:
    previous_cache = previous[0].parent
    print('Restoring previous successful stage:', previous_cache)
    shutil.copytree(previous_cache, CACHE, dirs_exist_ok=True)

state_file=CACHE/'_completed.json'
completed=json.loads(state_file.read_text()) if state_file.exists() else []
required=STAGES[:STAGES.index(STAGE)]
assert completed==required, (f'Expected completed stages {required}, got {completed}. '
                             'Attach the correct previous stage output, or start with check_data.')

if STAGE=='check_data':
    import pandas as pd
    candidates=[]
    for p in INPUT.rglob('train.csv'):
        if (p.parent/'test.csv').exists() and (p.parent/'sample_submission.csv').exists():
            candidates.append(p.parent)
    assert candidates, 'Connect the original competition train.csv / test.csv / sample_submission.csv'
    # Prefer Kaggle competition input instead of a notebook output containing copies.
    candidates.sort(key=lambda p: (0 if 'competitions' in p.parts else 1,str(p)))
    folder=candidates[0]
    for name in ('train.csv','test.csv','sample_submission.csv'):
        shutil.copy2(folder/name,CACHE/name)
    tr=pd.read_csv(CACHE/'train.csv',dtype={'id':str,'query_id':str})
    te=pd.read_csv(CACHE/'test.csv',dtype={'id':str,'query_id':str})
    ss=pd.read_csv(CACHE/'sample_submission.csv',dtype={'id':str})
    assert len(tr)==31530 and len(te)==13507
    assert tr.id.is_unique and te.id.is_unique and ss.id.is_unique
    assert set(ss.id)==set(te.id) and not (set(tr.query_id)&set(te.query_id))
    print('Rows:', len(tr),len(te),'groups:',tr.query_id.nunique(),te.query_id.nunique())
else:
    for name in ('train.csv','test.csv','sample_submission.csv'):
        assert (CACHE/name).exists(), f'Missing previous cached competition file: {name}'

for k,v in {'AMML_INPUT':str(CACHE),'AMML_OUT':str(CACHE),'AMML_OUTPUT':str(CACHE),
            'AMML_TEMP':'/kaggle/working/amml_repro_temp', 'USE_TF':'0','USE_FLAX':'0',
            'TOKENIZERS_PARALLELISM':'false','OMP_NUM_THREADS':'2','MKL_NUM_THREADS':'2',
            'OPENBLAS_NUM_THREADS':'2','MKL_NUM_THREADS':'2',
            'AMML_V6_VIEWS_ONLY':'1'}.items(): os.environ[k]=v

# Avoid replacing the working CUDA-enabled Kaggle PyTorch build. Install the
# recorded direct Python dependencies only when their versions do not match.
PINS = {'torch': '2.11.0', 'transformers': '4.51.3', 'accelerate': '1.6.0', 'peft': '0.15.2', 'bitsandbytes': '0.48.2', 'sentencepiece': '0.2.2', 'einops': '0.8.1', 'numpy': '2.1.3', 'pandas': '2.3.3', 'scipy': '1.16.3', 'scikit-learn': '1.6.1', 'safetensors': '0.8.0', 'huggingface-hub': '0.36.2', 'tokenizers': '0.21.4', 'tqdm': '4.67.3'}
if INSTALL_DEPENDENCIES:
    from importlib.metadata import version,PackageNotFoundError
    needed=[]
    for package,want in PINS.items():
        try: have=version(package)
        except PackageNotFoundError:have=None
        if package=='torch':
            assert have and have.split('+')[0]==want, (
                f'Expected Kaggle CUDA PyTorch {want}, found {have}. '
                'Use the original Kaggle GPU container; do not replace torch with CPU-only builds.')
        elif have!=want:
            needed.append(f'{package}=={want}')
    if needed:
        subprocess.run([sys.executable,'-m','pip','install','-q',
                        '--only-binary=sentencepiece',*needed],check=True)
print('Selected stage:',STAGE,'| prior completed:',completed)


## Реализация финальных обучающихся компонентов

Ниже сохранены функции, использованные в исходных успешных версиях. Они записываются во временную папку Kaggle **при каждом запуске**, но не являются отдельными файлами репозитория. Запуск экспериментов и подбор по Public LB в этом ноутбуке отсутствуют.

In [ ]:
%%writefile /kaggle/working/amml_repro_cache/solution_v5_worker.py
# AMML v5: independent Qwen3 4B / 8B QLoRA workers, Linux / Kaggle T4 x2.
# Notebook installs dependencies before launching this script twice.
import argparse, os, time, math, gc, re, html, unicodedata, random, json, platform
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

OUT=Path(os.environ.get('AMML_OUT','/kaggle/working'))
ROOT=Path(os.environ.get('AMML_INPUT','/kaggle/input'))
SPLIT_SEED=2028
VAL_FRAC=.15

def locate(name, required=True, aliases=()):
    files=sorted(ROOT.rglob('*.csv'))
    for candidate in (name,*aliases):
        matches=[p for p in files if p.name==candidate]
        if len(matches)>1: raise ValueError(f'Multiple {candidate}: {matches}')
        if matches: return matches[0]
    if name in ('train.csv','test.csv','sample_submission.csv'):
        matches=[p for p in files if re.fullmatch(re.escape(Path(name).stem)+r' \(\d+\)\.csv',p.name)]
        if len(matches)==1: return matches[0]
        if len(matches)>1: raise ValueError(f'Multiple {name}: {matches}')
    if required: raise FileNotFoundError(name)
    return None

def read_data():
    train=pd.read_csv(locate('train.csv'),dtype={'id':str,'query_id':str})
    test=pd.read_csv(locate('test.csv'),dtype={'id':str,'query_id':str})
    p=locate('sample_submission.csv',False)
    sample=pd.read_csv(p,dtype={'id':str}) if p else test[['id']].copy()
    assert train.id.is_unique and test.id.is_unique and sample.id.is_unique
    assert set(sample.id)==set(test.id) and set(train.relevance)=={0,1}
    assert not set(train.query_id)&set(test.query_id)
    tr,va=next(GroupShuffleSplit(n_splits=1,test_size=VAL_FRAC,random_state=SPLIT_SEED).split(train,groups=train.query_id))
    return train,test,sample,tr,va

def clean(s):
    if pd.isna(s): return ''
    s=unicodedata.normalize('NFKC',html.unescape(str(s)))
    return re.sub(r'\s+',' ',s.replace('\u0301','').replace('\u00ad','').replace('\u200b','')).strip()

def prepare(train,test):
    df=pd.concat([train.drop(columns='relevance'),test],ignore_index=True)
    for c in ['query','title','text']: df[c]=df[c].map(clean)
    known=df.loc[df.title.ne('') & df.article_id.notna(),['article_id','title']].drop_duplicates()
    unique=known.loc[~known.article_id.duplicated(keep=False)].set_index('article_id').title
    missing=df.title.eq('')
    df.loc[missing,'title']=df.loc[missing,'article_id'].map(unique).fillna('')
    print('Restored titles:',int(missing.sum()-df.title.eq('').sum()),flush=True)
    return df

def ndcg(frame,scores):
    a=frame[['id','query_id','relevance']].copy(); a['score']=np.asarray(scores,dtype=float)
    a=a.sort_values(['query_id','score','id'],ascending=[True,False,True])
    d=1/np.log2(np.arange(2,12)); rank=a.groupby('query_id',sort=False).cumcount().to_numpy()
    a['gain']=a.relevance.to_numpy()*np.where(rank<10,d[np.minimum(rank,9)],0.)
    dcg=a.groupby('query_id').gain.sum(); counts=a.groupby('query_id').relevance.sum().astype(int)
    ideal=np.r_[0.,np.cumsum(d)][np.minimum(counts,10)]
    return float(np.divide(dcg.to_numpy(),ideal,out=np.zeros(len(ideal)),where=ideal>0).mean())

def group_z(pred,qids):
    s=pd.Series(np.asarray(pred,dtype=float)); g=s.groupby(np.asarray(qids),sort=False)
    return ((s-g.transform('mean'))/(g.transform('std').fillna(0)+1e-6)).clip(-5,5).to_numpy()

def atomic_csv(df,path):
    tmp=path.with_suffix('.tmp'); df.to_csv(tmp,index=False); os.replace(tmp,path)

def read_prediction(path,ids):
    d=pd.read_csv(path,dtype={'id':str}); assert d.id.is_unique and set(d.id)==set(ids),path
    p=d.set_index('id').loc[np.asarray(ids),'prediction'].to_numpy(dtype=float)
    assert np.isfinite(p).all(),path
    return p

CONFIG={
 '4b':dict(model='Qwen/Qwen3-Reranker-4B',seed=5301,maxlen=640,batch=4,evalbatch=8,accum=4,layers=16,r=16,lr=3e-5,epochs=2,neg_limit=0,hard_limit=3),
 '8b':dict(model='Qwen/Qwen3-Reranker-8B',seed=5302,maxlen=512,batch=2,evalbatch=4,accum=8,layers=10,r=16,lr=4e-5,epochs=1,neg_limit=3,hard_limit=0),
}
INSTRUCTIONS={
 '4b':'Given a Russian web search question, identify passages that provide direct factual evidence for the answer. A passage about the same topic without evidence for the answer is not relevant. A relevant passage may contain the answer in an inflected form or paraphrase.',
 '8b':'Given a Russian question and a passage from an encyclopedia, decide if the passage contains information that answers the exact question. Match the requested person, entity, date, event or attribute; reject passages that merely mention related concepts.',
}

def train_worker(tag,hours,run_id,fixed_batches=None):
    import torch
    import torch.nn.functional as F
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, get_cosine_schedule_with_warmup
    from peft import LoraConfig,get_peft_model,get_peft_model_state_dict,set_peft_model_state_dict
    from tqdm import tqdm
    cfg=CONFIG[tag].copy(); start=time.monotonic(); deadline=start+hours*3600
    work=OUT/f'v5_{run_id}_{tag}'; work.mkdir(parents=True,exist_ok=True)
    seed=cfg['seed']; random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.set_num_threads(2)
    assert torch.cuda.is_available() and torch.cuda.device_count()==1,'Launch each worker with CUDA_VISIBLE_DEVICES set to one GPU.'
    device=torch.device('cuda:0'); dtype=torch.float16
    print(tag,platform.system(),torch.__version__,torch.cuda.get_device_name(0),cfg,flush=True)
    train,test,sample,tr,va=read_data(); df=prepare(train,test); n=len(train); y=train.relevance.to_numpy(np.float32)
    assert np.isclose(ndcg(train.iloc[va],y[va]),1)
    groups=[g.index.to_numpy(np.int64) for _,g in train.iloc[tr].groupby('query_id',sort=False)]
    # Lexical hardness is computed only from observed texts. No article/document IDs as model inputs.
    def terms(s): return set(re.findall(r'[а-яёa-z0-9]+',s.lower().replace('ё','е')))
    hardness=np.zeros(n,dtype=np.float32)
    for i in tr:
        q=terms(df.at[i,'query']); t=terms(df.at[i,'title']); d=terms(df.at[i,'text']); denom=max(len(q),1)
        hardness[i]=(len(q&d)+.75*len(q&t))/denom
    tok=AutoTokenizer.from_pretrained(cfg['model'],use_fast=True,revision={'4b':'22e683669bc0f0bd69640a1354a6d0aebcfeede5','8b':'77d193c791ed757ca307ee72715aa132723da912'}[tag])
    if tok.pad_token_id is None: tok.pad_token=tok.eos_token
    tok.padding_side='right'
    def encode(s): return tok.encode(s,add_special_tokens=False)
    prefix=encode('<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and the Instruct provided. Note that the answer can only be "yes" or "no".<|im_end|>\n<|im_start|>user\n')
    suffix=encode('<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n')
    sep=encode('\n...\n'); passage=encode('\nPassage: ')
    yes=tok.convert_tokens_to_ids('yes'); no=tok.convert_tokens_to_ids('no'); assert yes!=no and yes is not None and no is not None
    enc=[]
    # 4B keeps more context. Both always preserve the answer marker, query and title.
    for q,t,d in tqdm(df[['query','title','text']].itertuples(index=False,name=None),total=len(df),desc=f'{tag} tokenize',mininterval=30):
        header=encode(f'<Instruct>: {INSTRUCTIONS[tag]}\n<Query>: {q}\n<Document>: Title: ')+encode(t)[:80]+passage
        budget=cfg['maxlen']-len(prefix)-len(suffix)-len(header); assert budget>32
        text=encode(d)
        if len(text)>budget:
            front=int(.75*(budget-len(sep))); back=budget-len(sep)-front
            text=text[:front]+sep+text[-back:] if back else text[:front]+sep
        enc.append(prefix+header+text+suffix)
    lengths=np.array([len(x) for x in enc]); assert lengths.max()<=cfg['maxlen']
    del df; gc.collect()
    quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=dtype)
    base=AutoModelForCausalLM.from_pretrained(cfg['model'],revision={'4b':'22e683669bc0f0bd69640a1354a6d0aebcfeede5','8b':'77d193c791ed757ca307ee72715aa132723da912'}[tag],quantization_config=quant,torch_dtype=dtype,device_map={'':0},attn_implementation='sdpa',low_cpu_mem_usage=True)
    base.config.use_cache=False
    # Frozen embeddings / LM head stay fp16 to avoid redundant fp32 vocabulary matrices (especially 8B).
    # Compute only yes/no logits: no [batch, length, vocabulary] allocation.
    head=(base.lm_head.weight[yes].detach().float()-base.lm_head.weight[no].detach().float()).clone()
    bias=base.lm_head.bias; head_bias=(bias[yes]-bias[no]).detach().float().clone() if bias is not None else torch.tensor(0.,device=device)
    for p in base.parameters(): p.requires_grad_(False)
    # Keep normalization parameters fp32; do not upcast the enormous frozen embeddings.
    for name,p in base.named_parameters():
        if 'norm' in name and p.is_floating_point(): p.data=p.data.float()
    model=get_peft_model(base,LoraConfig(task_type='CAUSAL_LM',r=cfg['r'],lora_alpha=2*cfg['r'],lora_dropout=.05,bias='none',target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],layers_to_transform=list(range(base.config.num_hidden_layers-cfg['layers'],base.config.num_hidden_layers)),layers_pattern='layers'))
    model.print_trainable_parameters(); params=[p for p in model.parameters() if p.requires_grad]; assert params
    # Explicit float32 adapters + GradScaler for T4 fp16 stability.
    for p in params: p.data=p.data.float()
    def pack(idx):
        b=tok.pad([{'input_ids':enc[int(i)],'attention_mask':[1]*lengths[int(i)]} for i in idx],padding=True,pad_to_multiple_of=8,return_tensors='pt')
        return {k:v.to(device) for k,v in b.items()}
    def forward(idx):
        b=pack(idx)
        with torch.autocast('cuda',dtype=dtype):
            h=model.get_base_model().model(**b,use_cache=False,return_dict=True).last_hidden_state
        last=b['attention_mask'].sum(1)-1
        return h[torch.arange(len(idx),device=device),last].float()@head+head_bias
    model.eval()
    with torch.inference_mode():
        b=pack(va[:2]); last=b['attention_mask'].sum(1)-1
        with torch.autocast('cuda',dtype=dtype):
            h=model.get_base_model().model(**b,use_cache=False).last_hidden_state
            logits=model.get_base_model().lm_head(h[torch.arange(2,device=device),last]).float()
        ref=logits[:,yes]-logits[:,no]; got=forward(va[:2]); assert torch.allclose(ref,got,atol=.3,rtol=.04),(ref,got)
        del b,h,logits,ref,got
    def ranking_loss(s,l):
        bce=F.binary_cross_entropy_with_logits(s,l,pos_weight=s.new_tensor(2.))
        mask=l[:,None]>l[None,:]
        if not mask.any(): return bce
        # LambdaRank-inspired weight on available within-query pairs; ranks are local to the microbatch.
        with torch.no_grad():
            rank=torch.argsort(torch.argsort(s,descending=True,stable=True),stable=True)
            discount=1/torch.log2(rank.float()+2)
            weight=(discount[:,None]-discount[None,:]).abs()[mask].clamp_min(.05)
        delta=(s[:,None]-s[None,:])[mask]
        pair=(F.softplus(-delta)*weight).sum()/weight.sum()
        return .65*bce+.35*pair
    def probe(bs):
        ii=tr[np.argsort(lengths[tr])[-bs:]]; model.train(); loss=ranking_loss(forward(ii),torch.as_tensor(y[ii],device=device)); loss.backward()
        assert any(p.grad is not None and torch.isfinite(p.grad).all() and p.grad.abs().sum()>0 for p in params)
    while True:
        failed=False
        try: probe(cfg['batch'])
        except torch.cuda.OutOfMemoryError: failed=True
        finally:
            for p in params: p.grad=None
            gc.collect(); torch.cuda.empty_cache()
        if not failed: break
        if cfg['batch']<=2: raise RuntimeError('OOM at batch 2; this worker cannot finish on this GPU.')
        cfg['batch']//=2; cfg['accum']*=2; cfg['evalbatch']=min(cfg['evalbatch'],cfg['batch'])
        print('OOM preflight: batch reduced',cfg['batch'],flush=True)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    print('Memory probe OK, peak GiB',round(torch.cuda.max_memory_allocated()/2**30,2),flush=True)
    def predict(idx,desc):
        model.eval(); idx=np.asarray(idx,np.int64); order=np.argsort(lengths[idx],kind='stable'); result=np.empty(len(idx),np.float32)
        k=0; bs=cfg['evalbatch']
        with torch.inference_mode():
            bar=tqdm(total=len(order),desc=desc,mininterval=30)
            while k<len(order):
                p=order[k:k+bs]; failed=False
                try: v=forward(idx[p]).cpu().numpy()
                except torch.cuda.OutOfMemoryError: failed=True
                if failed:
                    if bs<=1: raise RuntimeError('Inference OOM at batch 1')
                    bs=max(1,bs//2); cfg['evalbatch']=bs; gc.collect(); torch.cuda.empty_cache(); continue
                result[p]=v; k+=len(p); bar.update(len(p))
            bar.close()
        assert np.isfinite(result).all(); return result
    # First 4B epoch covers every row. 8B uses all positives + diverse negatives.
    # Second 4B epoch emphasizes negatives scored too highly in epoch 1.
    def batches(epoch,cache=None):
        rng=np.random.default_rng(seed+epoch); result=[]
        lim=cfg['neg_limit'] if epoch==0 else cfg['hard_limit']
        rank_score=hardness if cache is None else np.where(np.isfinite(cache),cache,hardness)
        for idx in groups:
            pos=list(rng.permutation(idx[y[idx]==1])); neg=list(rng.permutation(idx[y[idx]==0])); assert pos and neg
            if lim and len(neg)>lim:
                # two hard negatives and one random negative for diversity
                neg=sorted(neg,key=lambda i:rank_score[i],reverse=True)
                hard=neg[:max(1,lim-1)]; rest=neg[len(hard):]; neg=hard+list(rng.choice(rest,size=lim-len(hard),replace=False))
                rng.shuffle(neg)
            mixed=[]
            while pos and neg: mixed.extend([pos.pop(),neg.pop()])
            tail=pos+neg; rng.shuffle(tail); mixed.extend(tail)
            result.extend([np.asarray(mixed[i:i+cfg['batch']],np.int64) for i in range(0,len(mixed),cfg['batch'])])
        # Shuffle blocks for SGD, sort only inside blocks to limit padding without a global length curriculum.
        rng.shuffle(result)
        for j in range(0,len(result),64): result[j:j+64]=sorted(result[j:j+64],key=lambda a:lengths[a].max())
        return result
    cache=np.full(n,np.nan,np.float32); first=batches(0)
    if tag=='4b': assert np.array_equal(np.sort(np.concatenate(first)),np.sort(tr))
    opt=torch.optim.AdamW(params,lr=cfg['lr'],weight_decay=.01,foreach=False)
    scaler=torch.amp.GradScaler('cuda',enabled=True)
    best=-1.; best_state=None; best_val=None; good_steps=0; history=[]; completed=[]; stop=False
    def validate(epoch,j):
        nonlocal best,best_state,best_val
        t0=time.monotonic(); p=predict(va,f'{tag} validation {epoch+1}:{j}'); score=ndcg(train.iloc[va],p)
        history.append(dict(epoch=epoch+1,microbatch=j,ndcg=score)); print('VALID',tag,history[-1],flush=True)
        if score>best:
            best=score; best_val=p.copy(); best_state={k:v.detach().cpu().clone() for k,v in get_peft_model_state_dict(model).items()}
            tmp=work/'best_adapter.tmp'; torch.save(best_state,tmp); os.replace(tmp,work/'best_adapter.pt')
            atomic_csv(pd.DataFrame({'id':train.id.iloc[va].to_numpy(),'prediction':p}),work/'validation.csv')
        model.train(); return time.monotonic()-t0
    # Checkpoints at 1/3, 2/3 and the end, and before a soft deadline.
    # The deadline is a safety cap, not a guarantee: a long inference/download can still exceed it.
    for epoch in range(cfg['epochs'] if fixed_batches is None else len(fixed_batches)):
        bs=first if epoch==0 else batches(epoch,cache)
        if fixed_batches is not None:
            assert 0<fixed_batches[epoch]<=len(bs) and (fixed_batches[epoch]%cfg['accum']==0 or fixed_batches[epoch]==len(bs))
        if epoch:
            # Optional hard stage only if measured time leaves room for validation and full test inference.
            eta=(time.monotonic()-epoch_start)/max(len(first),1)*len(bs)
            if fixed_batches is None and time.monotonic()+eta+max(3600,4*last_val_secs)>deadline:
                print('Skipping hard epoch, estimated time exceeds budget.',flush=True); break
            for pg in opt.param_groups: pg['lr']=cfg['lr']*.45; pg['initial_lr']=pg['lr']
        else: epoch_start=time.monotonic()
        total_updates=math.ceil(len(bs)/cfg['accum']); sched=get_cosine_schedule_with_warmup(opt,max(1,int(total_updates*.05)),total_updates)
        # Round validation points to optimizer boundaries.
        checkpoints={min(len(bs),math.ceil(len(bs)*f/cfg['accum'])*cfg['accum']) for f in (1/3,2/3,1)}
        last_val_secs=600 if epoch==0 else last_val_secs
        model.train(); opt.zero_grad(set_to_none=True); done=0
        bar=tqdm(enumerate(bs),total=len(bs),desc=f'{tag} epoch {epoch+1}',mininterval=30)
        for j,idx in bar:
            denom=min(cfg['accum'],len(bs)-(j//cfg['accum'])*cfg['accum'])
            s=forward(idx); cache[idx]=s.detach().cpu().numpy(); l=torch.as_tensor(y[idx],device=device); loss=ranking_loss(s,l)
            if not torch.isfinite(loss): raise FloatingPointError(f'{tag}: nonfinite loss')
            scaler.scale(loss/denom).backward(); done=j+1
            if done%cfg['accum']==0 or done==len(bs):
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params,1.)
                old_scale=scaler.get_scale(); scaler.step(opt); scaler.update()
                if scaler.get_scale()>=old_scale: good_steps+=1; sched.step()
                opt.zero_grad(set_to_none=True)
                # Reserve at least 1h, or 4 validation-inference times + 15m, for test inference/save.
                reserve=max(3600,4*last_val_secs+900)
                stop=(time.monotonic()+reserve>=deadline if fixed_batches is None else done==fixed_batches[epoch] and done<len(bs))
                if done in checkpoints or stop:
                    last_val_secs=validate(epoch,done)
                if stop: print('Soft deadline: closing training at',done,flush=True); break
        bar.close(); completed.append(done)
        if stop: break
    assert best_state is not None and good_steps>0
    set_peft_model_state_dict(model,best_state); model.eval()
    # Standard adapter directory (no frozen embeddings/base checkpoint).
    model.save_pretrained(work/'adapter',save_embedding_layers=False)
    pred=predict(np.arange(n,n+len(test)),f'{tag} TEST')
    atomic_csv(pd.DataFrame({'id':test.id,'prediction':pred}),work/'test.csv')
    meta={'tag':tag,'config':cfg,'split_seed':SPLIT_SEED,'completed_microbatches':completed,'successful_updates':good_steps,'best_validation':best,'history':history,'elapsed_hours':(time.monotonic()-start)/3600,'model_commit':getattr(base.config,'_commit_hash',None)}
    from importlib.metadata import version
    meta['versions']={k:version(k) for k in ['torch','numpy','pandas','scikit-learn','transformers','accelerate','peft','bitsandbytes','tokenizers','huggingface-hub','safetensors','tqdm','scipy']}
    (work/'run.json').write_text(json.dumps(meta,indent=2)); print('FINISHED',tag,meta,flush=True)

def combine(run_id):
    train,test,sample,tr,va=read_data(); valid=train.iloc[va]; qv=valid.query_id; qt=test.query_id
    val={}; preds={}; scores={}
    for tag in CONFIG:
        work=OUT/f'v5_{run_id}_{tag}'
        if not all((work/p).exists() for p in ['validation.csv','test.csv','run.json']):
            print('Missing completed worker:',tag); continue
        val[tag]=group_z(read_prediction(work/'validation.csv',valid.id),qv)
        preds[tag]=group_z(read_prediction(work/'test.csv',test.id),qt)
        scores[tag]=ndcg(valid,val[tag]); print(tag,'holdout:',scores[tag])
    if not preds: raise RuntimeError('No completed neural worker. Inspect logs; do not submit a partial file.')
    def save(name,p):
        result=sample[['id']].merge(pd.DataFrame({'id':test.id,'prediction':p}),on='id',how='left',validate='one_to_one')
        assert len(result)==len(test) and np.isfinite(result.prediction).all(); atomic_csv(result,OUT/name); print('Saved',name)
    for tag,p in preds.items(): save(f'submission_v5_{tag}.csv',p)
    # Small, predeclared ensemble grid; select only on validation, never on Public LB.
    candidates=[({tag:1.},scores[tag]) for tag in preds]
    if len(preds)==2:
        for w in (.25,.5,.75):
            weights={'4b':w,'8b':1-w}; score=ndcg(valid,sum(weights[k]*val[k] for k in weights)); candidates.append((weights,score))
    best_new=max(candidates,key=lambda a:a[1]); neww,newscore=best_new
    new_val=sum(neww[k]*val[k] for k in neww); new_pred=sum(neww[k]*preds[k] for k in neww)
    save('submission_v5_only.csv',new_pred)
    atomic_csv(pd.DataFrame({'id':valid.id,'prediction':new_val}),OUT/'v5_validation.csv')
    oldp=locate('best_0938300.csv',False,('submission_v4_only.csv',)); oldv=locate('v4_validation.csv',False)
    main=new_pred; choice={'weights':neww,'holdout':newscore}
    if oldp:
        op=group_z(read_prediction(oldp,test.id),qt)
        if oldv:
            ov=group_z(read_prediction(oldv,valid.id),qv)
            # Only comparable when v4_validation is from the same original v4 run and split.
            opts=[(w,ndcg(valid,(1-w)*ov+w*new_val)) for w in (0.,.25,.5,.75,1.)]
            print('Old/new validation blend grid:',opts)
            w,s=max(opts,key=lambda a:a[1]); main=(1-w)*op+w*new_pred
            choice={'old_v4_weight':1-w,'new_weight':w,'new_internal_weights':neww,'holdout':s}
            if w==0: print('Validation prefers old v4. Main submission is unchanged; v5_only remains available.')
            save('submission_v5_blend.csv',main)
        else:
            # Fixed hypothesis, not optimized; keep the validation-selected new-only solution as main.
            save('submission_v5_blend.csv',.5*op+.5*new_pred)
            print('No v4_validation.csv: 50/50 blend is experimental; no validation evidence for its weight.')
    save('submission.csv',main)
    report={'candidates':[(w,float(s)) for w,s in candidates],'selected_new':{'weights':neww,'holdout':newscore},'main':choice}
    (OUT/'v5_metrics.json').write_text(json.dumps(report,indent=2)); print(json.dumps(report,indent=2))

if __name__=='__main__':
    p=argparse.ArgumentParser(); p.add_argument('--worker',choices=['4b','8b','combine'],required=True); p.add_argument('--hours',type=float,default=8.5); p.add_argument('--run-id',default='run1'); p.add_argument('--fixed-batches',default=None,help='Comma-separated microbatch counts from run.json for exact training schedule'); a=p.parse_args()
    OUT.mkdir(parents=True,exist_ok=True)
    if a.worker=='combine': combine(a.run_id)
    else: train_worker(a.worker,a.hours,a.run_id,None if a.fixed_batches is None else [int(x) for x in a.fixed_batches.split(',')])


In [ ]:
%%writefile /kaggle/working/amml_repro_cache/solution_v6_worker.py
# AMML v6 — refine the already trained v5 Qwen adapters, Linux / Kaggle T4 x2.
# Only competition data and the user's own v5 training artifacts are used.
import argparse, os, time, math, gc, re, html, unicodedata, random, json, platform
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

ROOT=Path(os.environ.get('AMML_INPUT','/kaggle/input'))
OUT=Path(os.environ.get('AMML_OUT','/kaggle/working'))
SEED=2028
TAGS=('4b','8b')
CONFIG={
 '4b':dict(seed=6301,maxlen=768,evalbatch=8,lr=5e-6,mine_queries=1900,pairs=1500,accum=8),
 '8b':dict(seed=6302,maxlen=640,evalbatch=4,lr=6e-6,mine_queries=1900,pairs=1500,accum=8),
}
INSTRUCTIONS={
 '4b':'Given a Russian web search question, identify a passage that provides direct factual evidence for the exact answer. Pay attention to the requested entity, property, date or quantity. Accept a paraphrase or an implicit answer supported by the passage; reject merely topical matches.',
 '8b':'Определи, содержит ли фрагмент достоверную информацию, позволяющую ответить именно на данный вопрос на русском языке. Учитывай заголовок как контекст, различай одноимённые сущности, даты, количественные значения и запрошенные свойства. Простого совпадения темы недостаточно; ответ может быть выражен другими словами.',
}

def atomic_csv(frame,path):
    path=Path(path); tmp=path.with_suffix('.tmp'); frame.to_csv(tmp,index=False); os.replace(tmp,path)

def json_save(obj,path):
    path=Path(path); tmp=path.with_suffix('.tmp'); tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2)); os.replace(tmp,path)

def locate_data(stem,required_cols,optional=False):
    # v5 worker folders contain another test.csv; discriminate by header, not basename alone.
    found=[]
    for p in sorted(ROOT.rglob('*.csv')):
        if not re.fullmatch(re.escape(stem)+r'( \(\d+\))?\.csv',p.name): continue
        try: cols=set(pd.read_csv(p,nrows=0).columns)
        except Exception: continue
        if set(required_cols)<=cols and (stem!='test' or 'relevance' not in cols): found.append(p)
    if len(found)==1: return found[0]
    if not found and optional: return None
    raise RuntimeError(f'Expected one competition {stem}.csv; found {found}. Check connected inputs.')

def read_data():
    cols=['id','query_id','query','title','text','article_id']
    train=pd.read_csv(locate_data('train',cols+['relevance']),dtype={'id':str,'query_id':str})
    test=pd.read_csv(locate_data('test',cols),dtype={'id':str,'query_id':str})
    sp=locate_data('sample_submission',['id','prediction'],optional=True)
    sample=pd.read_csv(sp,dtype={'id':str}) if sp else test[['id']].copy()
    assert train.id.is_unique and test.id.is_unique and sample.id.is_unique
    assert set(sample.id)==set(test.id) and set(train.relevance)=={0,1}
    assert not set(train.query_id)&set(test.query_id)
    tr,va=next(GroupShuffleSplit(n_splits=1,test_size=.15,random_state=SEED).split(train,groups=train.query_id))
    return train,test,sample,tr,va

def clean(s):
    if pd.isna(s): return ''
    s=unicodedata.normalize('NFKC',html.unescape(str(s)))
    return re.sub(r'\s+',' ',s.replace('\u0301','').replace('\u00ad','').replace('\u200b','')).strip()

def prepare(train,test):
    df=pd.concat([train.drop(columns='relevance'),test],ignore_index=True)
    for c in ['query','title','text']: df[c]=df[c].map(clean)
    known=df.loc[df.title.ne('') & df.article_id.notna(),['article_id','title']].drop_duplicates()
    unique=known.loc[~known.article_id.duplicated(keep=False)].set_index('article_id').title
    missing=df.title.eq(''); df.loc[missing,'title']=df.loc[missing,'article_id'].map(unique).fillna('')
    print('Restored titles:',int(missing.sum()-df.title.eq('').sum()),flush=True)
    return df

def query_ndcg(frame,scores):
    a=frame[['id','query_id','relevance']].copy(); a['score']=np.asarray(scores,dtype=float)
    a=a.sort_values(['query_id','score','id'],ascending=[True,False,True]); d=1/np.log2(np.arange(2,12))
    rank=a.groupby('query_id',sort=False).cumcount().to_numpy(); a['gain']=a.relevance.to_numpy()*np.where(rank<10,d[np.minimum(rank,9)],0.)
    dcg=a.groupby('query_id').gain.sum(); counts=a.groupby('query_id').relevance.sum().astype(int)
    ideal=np.r_[0.,np.cumsum(d)][np.minimum(counts,10)]
    return np.divide(dcg.to_numpy(),ideal,out=np.zeros(len(ideal)),where=ideal>0)

def ndcg(frame,scores): return float(query_ndcg(frame,scores).mean())

def group_z(pred,qids):
    s=pd.Series(np.asarray(pred,dtype=float)); g=s.groupby(np.asarray(qids),sort=False)
    return ((s-g.transform('mean'))/(g.transform('std').fillna(0)+1e-6)).clip(-5,5).to_numpy()

def read_prediction(path,ids):
    d=pd.read_csv(path,dtype={'id':str}); assert d.id.is_unique and set(d.id)==set(ids),str(path)
    p=d.set_index('id').loc[np.asarray(ids),'prediction'].to_numpy(dtype=float); assert np.isfinite(p).all(),path
    return p

def get_sources(train,test,va):
    sources={}
    # Require the original worker outputs, not predictions from a different experiment.
    for p in sorted(ROOT.rglob('run.json')):
        try: meta=json.loads(p.read_text()); tag=meta.get('tag')
        except Exception: continue
        if tag not in TAGS or meta.get('split_seed')!=SEED: continue
        if meta.get('config',{}).get('model')!=f'Qwen/Qwen3-Reranker-{tag.upper()}': continue
        d=p.parent
        if not all((d/name).is_file() for name in ['validation.csv','test.csv']): continue
        adapter=d/'adapter'; pt=d/'best_adapter.pt'
        has_standard=(adapter/'adapter_config.json').is_file() and ((adapter/'adapter_model.safetensors').is_file() or (adapter/'adapter_model.bin').is_file())
        if tag in sources: raise RuntimeError(f'Multiple v5 worker directories for {tag}. Attach only one successful run.')
        val=read_prediction(d/'validation.csv',train.id.iloc[va]); pred=read_prediction(d/'test.csv',test.id)
        if abs(ndcg(train.iloc[va],val)-float(meta['best_validation']))>1e-6: raise ValueError(f'Validation mismatch for {d}')
        sources[tag]={'dir':d,'meta':meta,'adapter':adapter if has_standard else None,'pt':pt if pt.is_file() else None,'val':val,'test':pred}
    if not sources: raise RuntimeError('No v5 run directories found. Attach the preserved v5 output with run.json, validation.csv, test.csv, adapter/. Do not restart GPU training just to find them.')
    return sources

def anchor_from_sources(sources,valid,test):
    val={k:group_z(v['val'],valid.query_id) for k,v in sources.items()}; pred={k:group_z(v['test'],test.query_id) for k,v in sources.items()}
    # Reproduce the exact v5-only selection, including tie preference.
    options=[{k:1.} for k in TAGS if k in val]
    if len(val)==2: options += [{'4b':w,'8b':1-w} for w in (.25,.5,.75)]
    weights={'4b':0.5,'8b':0.5}  # Recorded from successful v6 run; fixed final recipe
    assert set(weights)==set(val),'Both fine-tuned Qwen versions are required'
    av=sum(weights[k]*val[k] for k in weights); at=sum(weights[k]*pred[k] for k in weights)
    # Check against the actual known v5 submission, when available.
    oldfiles=sorted(ROOT.rglob('submission_v5_only.csv'))
    if len(oldfiles)>1: raise RuntimeError('Multiple submission_v5_only.csv files; attach only one v5 run.')
    if oldfiles:
        original=read_prediction(oldfiles[0],test.id)
        if not np.allclose(original,at,atol=3e-6,rtol=3e-6):
            raise ValueError('v5 worker folders and submission_v5_only.csv are not from the same completed run.')
        at=original
    print('v5 anchor weights:',weights,'holdout:',ndcg(valid,av),flush=True)
    return av,at,weights

def preflight():
    train,test,_,tr,va=read_data(); sources=get_sources(train,test,va); anchor_from_sources(sources,train.iloc[va],test)
    enabled=[k for k in TAGS if k in sources and (sources[k]['adapter'] or sources[k]['pt'])]
    if not enabled: raise RuntimeError('No preserved adapter/ or best_adapter.pt. Do not spend GPU hours on this notebook until the saved weights are attached.')
    print('Train / validation groups:',train.query_id.iloc[tr].nunique(),train.query_id.iloc[va].nunique())
    for k,v in sources.items(): print(k,v['dir'],'weights:',v['adapter'] or v['pt'])
    return enabled

def focus_tokens(tokens,query_tokens,budget,sep):
    if len(tokens)<=budget: return tokens
    # Keep boundaries and a contiguous query-matched middle window. The separator is explicit.
    edge=min(100,max(16,budget//5)); mid=budget-2*edge-2*len(sep)
    if mid<=0: return tokens[:budget]
    lo,hi=edge,len(tokens)-edge-mid
    if hi<lo: return tokens[:budget]
    q=set(query_tokens); starts=list(range(lo,hi+1,max(16,mid//4)))+[hi]
    def quality(i):
        chunk=tokens[i:i+mid]; return len(q.intersection(chunk))+.005*sum(x in q for x in chunk)
    i=max(starts,key=quality)
    return tokens[:edge]+sep+tokens[i:i+mid]+sep+tokens[-edge:]

def select_pairs(frame,indices,scores,limit,seed):
    # Use only labeled TRAIN queries. Pick a low-scored true positive and high-scored false positive.
    rng=np.random.default_rng(seed); data=frame.iloc[indices][['id','query_id','relevance']].copy(); data['score']=np.asarray(scores)
    pairs=[]
    for _,g in data.groupby('query_id',sort=False):
        g=g.sort_values(['score','id'],ascending=[False,True]); y=g.relevance.to_numpy(); s=g.score.to_numpy(); ids=g.index.to_numpy()
        pos=np.flatnonzero(y==1); neg=np.flatnonzero(y==0)
        if not len(pos) or not len(neg): continue
        d=np.zeros(len(g)); d[:min(10,len(g))]=1/np.log2(np.arange(2,2+min(10,len(g))))
        ideal=(1/np.log2(np.arange(2,2+min(10,len(pos))))).sum()
        delta=np.abs(d[pos,None]-d[None,neg])/ideal; gap=s[neg][None,:]-s[pos][:,None]
        # Rank pairs by current logistic error times exact change in NDCG@10 for swapping positions.
        impact=(delta+.04)*np.logaddexp(0,np.clip(gap,-30,30))
        i,j=np.unravel_index(impact.argmax(),impact.shape)
        if rng.random()<.15: i,j=rng.integers(len(pos)),rng.integers(len(neg))
        pairs.append((int(ids[pos[i]]),int(ids[neg[j]]),float(delta[i,j]),float(impact[i,j])))
    pairs.sort(key=lambda x:x[3],reverse=True); limit=min(limit,len(pairs)); hard=min(int(.70*limit),len(pairs)); out=pairs[:hard]
    ids=rng.choice(len(pairs)-hard,size=limit-hard,replace=False) if limit>hard else []
    out += [pairs[hard+int(i)] for i in ids]; rng.shuffle(out)
    return out

def worker(tag,hours,run_id,fixed_pairs=None):
    import torch
    import torch.nn.functional as F
    from transformers import AutoTokenizer,AutoModelForCausalLM,BitsAndBytesConfig
    from peft import PeftModel,LoraConfig,get_peft_model,get_peft_model_state_dict,set_peft_model_state_dict
    from tqdm import tqdm
    cfg=CONFIG[tag].copy(); start=time.monotonic(); deadline=start+hours*3600
    train,test,sample,tr,va=read_data(); sources=get_sources(train,test,va); src=sources[tag]
    assert src['adapter'] or src['pt']; v5cfg=src['meta']['config']; revision=src['meta'].get('model_commit') or None
    anchor_val,_,_=anchor_from_sources(sources,train.iloc[va],test)
    seed=cfg['seed']; random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed); torch.set_num_threads(2)
    assert torch.cuda.is_available() and torch.cuda.device_count()==1
    device=torch.device('cuda:0'); dtype=torch.float16; work=OUT/f'v6_{run_id}_{tag}'; work.mkdir(parents=True,exist_ok=True)
    # Fixed held-out queries from v4/v5 are never used for gradient updates.
    rng=np.random.default_rng(seed); groups=[g.index.to_numpy(np.int64) for _,g in train.iloc[tr].groupby('query_id',sort=True)]
    chosen=rng.choice(len(groups),size=min(cfg['mine_queries'],len(groups)),replace=False); mine=np.concatenate([groups[j] for j in chosen]); n=len(train)
    testidx=np.arange(n,n+len(test)); needed=np.unique(np.r_[mine,va,testidx]); df=prepare(train,test)
    tok=AutoTokenizer.from_pretrained(v5cfg['model'],revision=revision,use_fast=True)
    if tok.pad_token_id is None: tok.pad_token=tok.eos_token
    tok.padding_side='right'
    def encode(s): return tok.encode(s,add_special_tokens=False)
    prefix=encode('<|im_start|>system\nJudge whether the Document meets the requirements based on the Query and the Instruct provided. Note that the answer can only be "yes" or "no".<|im_end|>\n<|im_start|>user\n')
    suffix=encode('<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'); sep=encode('\n...\n'); passage=encode('\nPassage: ')
    yes=tok.convert_tokens_to_ids('yes'); no=tok.convert_tokens_to_ids('no'); assert yes!=no and yes is not None and no is not None
    enc={}; lengths=np.zeros(len(df),dtype=np.int32)
    for i in tqdm(needed,desc=f'{tag} tokenize',mininterval=30):
        q,t,d=df.loc[i,['query','title','text']]; header=encode(f'<Instruct>: {INSTRUCTIONS[tag]}\n<Query>: {q}\n<Document>: Title: ')+encode(t)[:80]+passage
        budget=cfg['maxlen']-len(prefix)-len(suffix)-len(header); assert budget>32
        enc[int(i)]=prefix+header+focus_tokens(encode(d),encode(q),budget,sep)+suffix; lengths[i]=len(enc[int(i)])
    assert lengths.max()<=cfg['maxlen']; del df; gc.collect()
    quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=dtype)
    base=AutoModelForCausalLM.from_pretrained(v5cfg['model'],revision=revision,quantization_config=quant,torch_dtype=dtype,device_map={'':0},attn_implementation='sdpa',low_cpu_mem_usage=True)
    base.config.use_cache=False
    head=(base.lm_head.weight[yes].detach().float()-base.lm_head.weight[no].detach().float()).clone(); bias=base.lm_head.bias
    head_bias=(bias[yes]-bias[no]).detach().float().clone() if bias is not None else torch.tensor(0.,device=device)
    for p in base.parameters(): p.requires_grad_(False)
    for name,p in base.named_parameters():
        if 'norm' in name and p.is_floating_point(): p.data=p.data.float()
    if src['adapter']:
        model=PeftModel.from_pretrained(base,str(src['adapter']),is_trainable=True)
    else:
        model=get_peft_model(base,LoraConfig(task_type='CAUSAL_LM',r=v5cfg['r'],lora_alpha=2*v5cfg['r'],lora_dropout=.05,bias='none',target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],layers_to_transform=list(range(base.config.num_hidden_layers-v5cfg['layers'],base.config.num_hidden_layers)),layers_pattern='layers'))
        state=torch.load(src['pt'],map_location='cpu',weights_only=True); expected=get_peft_model_state_dict(model)
        assert set(state)==set(expected),'v5 adapter keys do not match'; set_peft_model_state_dict(model,state); del state,expected
    params=[p for p in model.parameters() if p.requires_grad]; assert params
    for p in params: p.data=p.data.float()
    model.print_trainable_parameters()
    def forward(idx):
        batch=tok.pad([{'input_ids':enc[int(i)],'attention_mask':[1]*int(lengths[i])} for i in idx],padding=True,pad_to_multiple_of=8,return_tensors='pt'); batch={k:v.to(device) for k,v in batch.items()}
        with torch.autocast('cuda',dtype=dtype): h=model.get_base_model().model(**batch,use_cache=False,return_dict=True).last_hidden_state
        last=batch['attention_mask'].sum(1)-1; return h[torch.arange(len(idx),device=device),last].float()@head+head_bias
    def predict(idx,desc):
        model.eval(); idx=np.asarray(idx,np.int64); order=np.argsort(lengths[idx],kind='stable'); result=np.empty(len(idx),np.float32); k=0; bs=cfg['evalbatch']
        with torch.inference_mode():
            with tqdm(total=len(order),desc=desc,mininterval=30) as bar:
                while k<len(order):
                    p=order[k:k+bs]; failed=False
                    try: v=forward(idx[p]).cpu().numpy()
                    except torch.cuda.OutOfMemoryError: failed=True
                    if failed:
                        if bs<=1: raise RuntimeError('Inference OOM at batch 1')
                        bs=max(1,bs//2); cfg['evalbatch']=bs; gc.collect(); torch.cuda.empty_cache(); continue
                    result[p]=v; k+=len(p); bar.update(len(p))
        assert np.isfinite(result).all(); return result
    def val_score(p):
        z=group_z(p,train.query_id.iloc[va]); return ndcg(train.iloc[va],.65*anchor_val+.35*z)
    def save_pair(name,val,pred):
        d=work/name; d.mkdir(exist_ok=True)
        atomic_csv(pd.DataFrame({'id':train.id.iloc[va].to_numpy(),'prediction':val}),d/'validation.csv')
        atomic_csv(pd.DataFrame({'id':test.id.to_numpy(),'prediction':pred}),d/'test.csv')
        json_save({'ndcg':ndcg(train.iloc[va],val),'anchor_blend_ndcg':val_score(val)},d/'complete.json')
    # Stage 1. Alternative prompt/context inference with the existing trained weights.
    t0=time.monotonic(); view_val=predict(va,f'{tag} view validation'); val_secs=time.monotonic()-t0
    print('VIEW',tag,'NDCG',ndcg(train.iloc[va],view_val),'blend',val_score(view_val),flush=True)
    t0=time.monotonic(); view_test=predict(testidx,f'{tag} view TEST'); test_secs=time.monotonic()-t0; save_pair('view',view_val,view_test)
    meta={'tag':tag,'source':str(src['dir']),'model_commit':revision,'config':cfg,'view_val_seconds':val_secs,'view_test_seconds':test_secs,'completed_pairs':0,'history':[]}
    json_save(meta,work/'run.json')
    # Stage 2. Mine difficult pairs using only training labels and scores from this model.
    # Guard reserves full test inference, up to two validations and a safety buffer.
    reserve=max(1800,1.30*test_secs+2.30*val_secs+600)
    estimated_mine=1.25*test_secs*len(mine)/len(testidx)
    if os.environ.get('AMML_V6_VIEWS_ONLY','1')=='1' or (fixed_pairs is None and time.monotonic()+reserve+estimated_mine+600>=deadline):
        print('Time guard: keeping completed view, no hard training.',flush=True); return
    mining=predict(mine,f'{tag} mine TRAIN'); pairs=select_pairs(train,mine,mining,cfg['pairs'],seed)
    assert pairs and set(x for p in pairs for x in p[:2])<=set(tr)
    # Store exact training pair indices as IDs for reproducibility, without using opaque IDs as features.
    atomic_csv(pd.DataFrame([{'positive_id':train.id.iloc[p],'negative_id':train.id.iloc[m],'delta_ndcg':w,'hardness':h} for p,m,w,h in pairs]),work/'mined_pairs.csv')
    if fixed_pairs is not None:
        assert 0<fixed_pairs<=len(pairs) and (fixed_pairs%cfg['accum']==0 or fixed_pairs==len(pairs)); target=fixed_pairs
    else: target=len(pairs)
    opt=torch.optim.AdamW(params,lr=cfg['lr'],weight_decay=.01,foreach=False); scaler=torch.amp.GradScaler('cuda',enabled=True)
    best_score=-np.inf; best_state=None; best_val=None; best_step=0; done=0; updates=0; opt.zero_grad(set_to_none=True)
    def validate(step):
        nonlocal best_score,best_state,best_val,best_step
        p=predict(va,f'{tag} hard validation {step}'); score=val_score(p)
        item={'pairs':step,'ndcg':ndcg(train.iloc[va],p),'anchor_blend_ndcg':score}; meta['history'].append(item); print('HARD',tag,item,flush=True)
        if score>best_score:
            best_score=score; best_val=p.copy(); best_step=step; best_state={k:v.detach().cpu().clone() for k,v in get_peft_model_state_dict(model).items()}
            tmp=work/'hard_best.tmp'; torch.save(best_state,tmp); os.replace(tmp,work/'hard_best_adapter.pt')
            atomic_csv(pd.DataFrame({'id':train.id.iloc[va].to_numpy(),'prediction':p}),work/'hard_best_validation.csv')
        model.train()
    model.train(); accum=cfg['accum']; checkpoint=max(accum,(len(pairs)//2//accum)*accum)
    # Each microbatch is one positive and one negative from the SAME query; use a true global-position swap weight computed during mining.
    # Keeping the weight detached is the LambdaRank surrogate approximation.
    for j,(pos,neg,delta,_) in enumerate(tqdm(pairs[:target],desc=f'{tag} hard pairs',mininterval=30)):
        denom=min(accum,target-(j//accum)*accum); s=forward([pos,neg]); labels=s.new_tensor([1.,0.]);
        weight=max(.10,min(2.,.25+3*delta)); loss=.25*F.binary_cross_entropy_with_logits(s,labels)+.75*weight*F.softplus(s[1]-s[0]);
        if not torch.isfinite(loss): raise FloatingPointError('Nonfinite loss')
        scaler.scale(loss/denom).backward(); done=j+1
        if done%accum==0 or done==target:
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params,1.); old=scaler.get_scale(); scaler.step(opt); scaler.update(); updates+=int(scaler.get_scale()>=old); opt.zero_grad(set_to_none=True)
            # Gradually decay learning rate while retaining a small final learning rate.
            for pg in opt.param_groups: pg['lr']=cfg['lr']*(.3+.7*(1-done/max(len(pairs),1)))
            remaining=deadline-time.monotonic(); stop=fixed_pairs is None and remaining<reserve
            if done==checkpoint or done==target or stop: validate(done)
            if stop: print('Time guard: finished after',done,'pairs',flush=True); break
    if best_state is None or not updates: print('No finite hard updates; retain view.',flush=True); return
    meta.update(completed_pairs=done,best_pair_step=best_step,successful_updates=updates)
    set_peft_model_state_dict(model,best_state); model.eval(); model.save_pretrained(work/'hard_adapter',save_embedding_layers=False)
    json_save(meta,work/'run.json')
    new_test=predict(testidx,f'{tag} hard TEST'); save_pair('hard',best_val,new_test)
    meta['elapsed_hours']=(time.monotonic()-start)/3600; json_save(meta,work/'run.json'); print('FINISHED',tag,meta,flush=True)

def combine(run_id):
    train,test,sample,tr,va=read_data(); valid=train.iloc[va]; sources=get_sources(train,test,va); av,at,aw=anchor_from_sources(sources,valid,test); val={'v5':av}; preds={'v5':at}
    def save(name,p):
        result=sample[['id']].merge(pd.DataFrame({'id':test.id,'prediction':p}),on='id',how='left',validate='one_to_one'); assert len(result)==len(test) and np.isfinite(result.prediction).all(); atomic_csv(result,OUT/name)
    for tag in TAGS:
        for stage in ('view','hard'):
            d=OUT/f'v6_{run_id}_{tag}'/stage
            if not all((d/f).is_file() for f in ['validation.csv','test.csv','complete.json']): continue
            k=f'{tag}_{stage}'; val[k]=group_z(read_prediction(d/'validation.csv',valid.id),valid.query_id); preds[k]=group_z(read_prediction(d/'test.csv',test.id),test.query_id)
            save(f'submission_v6_{k}.csv',preds[k]); print(k,ndcg(valid,val[k]),flush=True)
    # Restrict search: baseline, single new view/hard and small shared blends.
    # A view and its fine-tuned version are highly correlated. Select at most one per architecture by anchor blend.
    new=[]
    for tag in TAGS:
        keys=[k for k in val if k.startswith(tag+'_')]
        if keys: new.append(max(keys,key=lambda k:ndcg(valid,.65*av+.35*val[k])))
    alternatives=[{k:1.} for k in new]
    if len(new)==2: alternatives.append({new[0]:.65,new[1]:.35})
    options=[{'v5':1.}]
    for a in alternatives:
        for w in (.25,.5,1.): options.append({'v5':1-w,**{k:w*x for k,x in a.items()}})
    base_per=query_ndcg(valid,av); records=[]
    for w in options:
        p=sum(x*val[k] for k,x in w.items()); delta=query_ndcg(valid,p)-base_per; score=float(base_per.mean()+delta.mean()); se=float(delta.std(ddof=1)/np.sqrt(len(delta)))
        # Conservative validation selection: penalize uncertain improvements; not a confidence guarantee.
        records.append({'weights':w,'ndcg':score,'delta':float(delta.mean()),'paired_se':se,'selection_score':float(delta.mean()-.5*se)})
    best=next(r for r in records if r['weights']=={'v5':0.75,'4b_view':0.25}); bw=best['weights']; base=records[0]  # Preserved final Qwen v6 recipe
    # Do not replace the established solution for a negligible local advantage.
    assert bw=={'v5':0.75,'4b_view':0.25}  # Do not reselect by a reused validation split
    candidates=[r for r in records if r['weights'].get('v5',1.)<1.]
    if candidates:
        experimental=max(candidates,key=lambda r:r['selection_score']); save('submission_v6_candidate.csv',sum(x*preds[k] for k,x in experimental['weights'].items()))
    main=sum(x*preds[k] for k,x in bw.items()); save('submission.csv',main); save('submission_v6_selected.csv',main)
    atomic_csv(pd.DataFrame({'id':valid.id.to_numpy(),'prediction':sum(x*val[k] for k,x in bw.items())}),OUT/'v6_validation.csv')
    report={'v5_weights':aw,'v5_holdout':ndcg(valid,av),'individual_holdout':{k:ndcg(valid,p) for k,p in val.items()},'selected':best,'candidate':max(candidates,key=lambda r:r['selection_score']) if candidates else None,'grid':records}
    json_save(report,OUT/'v6_metrics.json'); print(json.dumps(report,indent=2),flush=True)
    if not new: print('No new worker inference completed. Inspect logs: only v5 was preserved.',flush=True)
    if bw=={'v5':1.}: print('Validation does not support replacing v5. submission.csv is the previous v5 solution.',flush=True)

if __name__=='__main__':
    p=argparse.ArgumentParser(); p.add_argument('--mode',required=True,choices=['preflight','4b','8b','combine']); p.add_argument('--hours',type=float,default=4.5); p.add_argument('--run-id',default='run1'); p.add_argument('--fixed-pairs',type=int,default=None); a=p.parse_args(); OUT.mkdir(parents=True,exist_ok=True)
    if a.mode=='preflight': preflight()
    elif a.mode=='combine': combine(a.run_id)
    else: worker(a.mode,a.hours,a.run_id,a.fixed_pairs)


In [ ]:
%%writefile /kaggle/working/amml_repro_cache/solution_fast.py
"""AMML restart: Jina reranker + multilingual extractive QA. Kaggle Linux T4 x2.
Only the competition CSVs and published pretrained checkpoints are used.
Run the notebook (or this file with --launch); no previous submissions/adapters.
"""
import os
os.environ.setdefault('USE_TF','0')
os.environ.setdefault('USE_FLAX','0')
os.environ.setdefault('TOKENIZERS_PARALLELISM','false')
os.environ.setdefault('OMP_NUM_THREADS','2')
os.environ.setdefault('MKL_NUM_THREADS','2')
os.environ.setdefault('OPENBLAS_NUM_THREADS','2')
import sys, argparse, time, json, math, re, html, unicodedata, random, traceback, subprocess
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, GroupKFold

ROOT=Path(os.environ.get('AMML_INPUT','/kaggle/input'))
OUT=Path(os.environ.get('AMML_OUTPUT','/kaggle/working'))
WORK=Path(os.environ.get('AMML_TEMP','/tmp/amml_fast_restart'))
SEED=731
MODEL_IDS={'rerank':'jinaai/jina-reranker-v2-base-multilingual',
           'qa':'timpal0l/mdeberta-v3-base-squad2',
           'fallback':'cross-encoder/mmarco-mMiniLMv2-L12-H384-v1'}
CONFIG={'rerank':dict(seed=7311,maxlen=288,top=6,lr=2e-5,epochs=2,groups=2,evalbatch=40),
        'qa':dict(seed=7312,maxlen=288,top=4,lr=1.5e-5,epochs=2,groups=2,evalbatch=32),
        'fallback':dict(seed=7313,maxlen=288,top=10,lr=2.5e-5,epochs=2,groups=3,evalbatch=64)}
# For exact reproduction, fill from fast_manifest.json, including selected mode/model/step count.
FIXED_BATCHES={'rerank':2586}  # Exactly the optimizer-batch count recorded by original successful Jina training
MODEL_REVISIONS={'rerank':'9cfeff2df7d40d1b78e75e5e9cebec92a99813c9'}
MODEL_OVERRIDE={}  # e.g. {'rerank': 'fallback'} if the actual successful run used MiniLM
STOPWORDS=set('когда сколько кто какой какая какие какое каких каких как где чем что почему зачем куда откуда был была были было является называется это в во на по для при из с со о об у и а но же ли не до после за к от его ее или который которая'.split())

def atomic_csv(df,path):
    path=Path(path); tmp=path.with_suffix('.tmp'); df.to_csv(tmp,index=False); os.replace(tmp,path)

def atomic_json(obj,path):
    path=Path(path); tmp=path.with_suffix('.tmp'); tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2),encoding='utf-8'); os.replace(tmp,path)

def locate(stem,n):
    override=os.environ.get('AMML_'+stem.upper())
    paths=[Path(override)] if override else sorted(set(ROOT.rglob(stem+'.csv')) | set(ROOT.rglob(stem+' (*)'+'.csv')))
    for p in paths:
        try:
            df=pd.read_csv(p,dtype={'id':str,'query_id':str,'article_id':str,'document_id':str})
            if len(df)==n and ('relevance' in df)==(stem=='train'): return df,p
        except Exception: continue
    raise FileNotFoundError(f'Cannot find competition {stem}.csv with {n} rows; candidates: {paths}')

def clean(x):
    if pd.isna(x): return ''
    s=unicodedata.normalize('NFKC',html.unescape(str(x)))
    s=s.replace('\u200b','').replace('\ufeff','')
    s=re.sub(r'</?(?:p|div|span|br|b|i|strong|em)(?:\s[^>]*)?/?>',' ',s,flags=re.I)
    return re.sub(r'\s+',' ',s).strip()

def clean_title(x):
    return re.sub(r'\s*[-–—|]\s*Википедия\s*$','',clean(x),flags=re.I).strip()

def words(x): return re.findall(r'[а-яёa-z0-9]+',str(x).lower().replace('ё','е'))

def keyset(x): return {w[:5] for w in words(x) if w not in STOPWORDS and len(w)>2}

def focus(q,t,maxchars=1500):
    if len(t)<=maxchars: return t
    # Retain the beginning and the most query-matching contiguous later window.
    # No label, external corpus, or answer lookup is used.
    qs=keyset(q)
    head=t[:420]
    width=maxchars-440
    starts=range(0,max(1,len(t)-width+1),max(160,width//3))
    opts=list(starts)+[max(0,len(t)-width)]
    def score(i):
        w=keyset(t[i:i+width]); return (len(w&qs),-i)
    best=max(opts,key=score)
    if best<=300: return t[:maxchars]
    return head+' … '+t[best:best+width]

def prepare():
    WORK.mkdir(parents=True,exist_ok=True); OUT.mkdir(parents=True,exist_ok=True)
    tr,p=locate('train',31530); te,p2=locate('test',13507); sample,p3=locate('sample_submission',13507)
    assert set(sample.id)==set(te.id) and sample.id.is_unique and tr.id.is_unique and te.id.is_unique
    assert not set(tr.query_id)&set(te.query_id)
    df=pd.concat([tr,te],ignore_index=True); n=len(tr)
    df['query']=df['query'].map(clean); df['text']=df['text'].map(clean)
    df['title_clean']=df['title'].map(clean_title)
    df['title_missing']=df.title_clean.eq('').astype(int)
    known=df.loc[df.title_clean.ne('')].groupby('article_id').title_clean.agg(['first','nunique'])
    known=known.loc[known['nunique'].eq(1),'first']
    mask=df.title_clean.eq('')
    df.loc[mask,'title_clean']=df.loc[mask,'article_id'].map(known).fillna('')
    print('Restored titles',int((mask&df.title_clean.ne('')).sum()),flush=True)
    df['doc_view']=[(title[:250]+'\n' if title else '')+focus(q,t) for q,t,title in zip(df['query'],df.text,df.title_clean)]
    a,v=next(GroupShuffleSplit(n_splits=1,test_size=.15,random_state=SEED).split(tr,groups=tr.query_id))
    assert not set(df.query_id.iloc[a])&set(df.query_id.iloc[v]); assert len(a)+len(v)==n
    test=np.arange(n,len(df)); df.to_pickle(WORK/'data.pkl'); sample[['id']].to_csv(WORK/'sample.csv',index=False)
    np.savez(WORK/'split.npz',train=a,valid=v,test=test)
    print('Data',p,p2,p3,'groups',df.query_id.iloc[a].nunique(),df.query_id.iloc[v].nunique(),flush=True)
    return df,a,v,test,sample[['id']]

def get_data():
    df=pd.read_pickle(WORK/'data.pkl'); z=np.load(WORK/'split.npz'); return df,z['train'],z['valid'],z['test'],pd.read_csv(WORK/'sample.csv',dtype={'id':str})

def group_positions(df,idx):
    sub=df.iloc[idx]; return [np.asarray(x,dtype=int) for x in sub.groupby('query_id',sort=False).indices.values()]

def ndcg_values(df,idx,scores):
    s=np.asarray(scores,float); assert s.shape==(len(idx),) and np.isfinite(s).all()
    sub=df.iloc[idx]; y=sub.relevance.to_numpy(); ids=sub.id.astype(str).to_numpy(); d=1/np.log2(np.arange(2,12)); vals=[]
    for g in group_positions(df,idx):
        order=np.lexsort((ids[g],-s[g])); k=min(len(g),10); ideal=float(d[:min(int(y[g].sum()),10)].sum())
        vals.append(float(np.dot(y[g[order[:k]]],d[:k])/ideal) if ideal else 0.)
    return np.asarray(vals)

def ndcg(df,idx,scores): return float(ndcg_values(df,idx,scores).mean())

def normalize(df,idx,scores):
    s=np.asarray(scores,dtype=np.float64); o=np.zeros(len(s))
    for g in group_positions(df,idx):
        x=s[g]; o[g]=np.clip((x-x.mean())/max(x.std(),1e-5),-4,4)
    return o

def submission(df,idx,scores,sample,path):
    assert np.isfinite(scores).all() and len(scores)==len(idx)
    out=sample.merge(pd.DataFrame({'id':df.id.iloc[idx].to_numpy(),'prediction':scores}),on='id',how='left',validate='one_to_one')
    assert len(out)==13507 and not out.prediction.isna().any(); atomic_csv(out,path)

def make_batches(df,a,enc,epoch,groups_per,seed):
    groups=[a[g] for g in group_positions(df,a)]
    rng=np.random.default_rng(seed+epoch); rng.shuffle(groups)
    # Length bucketing within randomized small pools: reduce padding without sorting by labels.
    batches=[]
    for j in range(0,len(groups),64):
        chunk=sorted(groups[j:j+64],key=lambda g:max(len(enc[int(i)]['input_ids']) for i in g))
        bag=[chunk[k:k+groups_per] for k in range(0,len(chunk),groups_per)]
        rng.shuffle(bag); batches.extend(bag)
    return batches

def rank_loss(scores,labels,lengths):
    import torch
    import torch.nn.functional as F
    losses=[]; offset=0
    for n in lengths:
        s=scores[offset:offset+n].float(); y=labels[offset:offset+n].float(); offset+=n
        bce=F.binary_cross_entropy_with_logits(s,y,pos_weight=s.new_tensor(2.0))
        # All judged candidates of a query are present in the batch.
        with torch.no_grad():
            ranks=torch.argsort(torch.argsort(s,descending=True,stable=True),stable=True)
            disc=torch.where(ranks<10,1/torch.log2(ranks.float()+2),0.)
            match=(y[:,None]>y[None,:]).float()
            ideal=(1/torch.log2(torch.arange(n,device=s.device).float()+2))*((torch.arange(n,device=s.device)<y.sum())&(torch.arange(n,device=s.device)<10))
            w=((disc[:,None]-disc[None,:]).abs()/ideal.sum().clamp_min(1e-6)+0.02)*match
        pair=(F.softplus(s[None,:]-s[:,None])*w).sum()/w.sum().clamp_min(1e-6)
        listwise=-(F.log_softmax(s,dim=0)*y).sum()/y.sum().clamp_min(1)
        losses.append(.44*bce+.46*pair+.10*listwise)
    return torch.stack(losses).mean()

def qa_score(start,end,mask):
    """Best valid contiguous answer span vs CLS no-answer, max width 32 tokens."""
    import torch
    import torch.nn.functional as F
    start=start.float(); end=end.float(); mask=mask.bool(); L=start.shape[1]
    # Restrict start/end to actual document tokens, not padding/question/special tokens.
    ends=F.pad(end.masked_fill(~mask,-1e4),(0,31),value=-1e4).unfold(1,32,1)
    pairs=start.masked_fill(~mask,-1e4).unsqueeze(-1)+ends
    best=pairs.flatten(1).max(dim=1).values
    null=start[:,0]+end[:,0]
    return (best-null)/2

def worker(tag,deadline,force_fallback=False):
    import torch
    from transformers import AutoTokenizer,AutoModelForSequenceClassification,AutoModelForQuestionAnswering
    mode='fallback' if force_fallback else MODEL_OVERRIDE.get(tag,tag); df,a,v,te,sample=get_data(); model=None
    assert torch.cuda.is_available(); torch.cuda.set_device(0)
    torch.set_num_threads(2)
    def load(mode):
        name=MODEL_IDS[mode]; kwargs={}
        if mode in MODEL_REVISIONS: kwargs['revision']=MODEL_REVISIONS[mode]
        else:
            from huggingface_hub import HfApi
            try: kwargs['revision']=HfApi().model_info(name,timeout=12).sha
            except Exception: pass
        tok=AutoTokenizer.from_pretrained(name,use_fast=True,**kwargs)
        if mode=='qa': m=AutoModelForQuestionAnswering.from_pretrained(name,torch_dtype=torch.float32,**kwargs)
        elif mode=='rerank':
            m=AutoModelForSequenceClassification.from_pretrained(name,trust_remote_code=True,use_flash_attn=False,torch_dtype=torch.float32,**kwargs)
        else: m=AutoModelForSequenceClassification.from_pretrained(name,torch_dtype=torch.float32,**kwargs)
        m._amml_revision=kwargs.get('revision') or getattr(m.config,'_commit_hash',None)
        return tok,m
    try: tok,model=load(mode)
    except Exception:
        if mode!='rerank': raise
        raise RuntimeError('The exact Jina checkpoint could not be loaded. Do not silently replace the final model.')
    tok.padding_side='right'
    cfg=CONFIG[mode]; seed=cfg['seed']; random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark=False
    # Freeze embeddings and lower blocks. This reduces backward cost and optimizer state.
    for p in model.parameters(): p.requires_grad_(False)
    base=model.base_model
    layers=getattr(base.encoder,'layers',None)
    if layers is None: layers=base.encoder.layer
    for layer in layers[-cfg['top']:]:
        for p in layer.parameters(): p.requires_grad_(True)
    head=model.qa_outputs if mode=='qa' else model.classifier
    for p in head.parameters(): p.requires_grad_(True)
    model.to('cuda'); params=[p for p in model.parameters() if p.requires_grad]
    print(tag,'actual model',MODEL_IDS[mode],'params',sum(p.numel() for p in params),'/',sum(p.numel() for p in model.parameters()),flush=True)
    enc=[]
    for start in range(0,len(df),512):
        sub=df.iloc[start:start+512]
        x=tok(sub['query'].tolist(),sub.doc_view.tolist(),truncation=True,max_length=cfg['maxlen'],padding=False)
        for j,ids in enumerate(x['input_ids']):
            z={k:x[k][j] for k in ['input_ids','attention_mask','token_type_ids'] if k in x}
            if mode=='qa': z['context_mask']=[int(i==1) for i in x.sequence_ids(j)]
            enc.append(z)
    def collate(indices):
        rows=[enc[int(i)] for i in indices]
        batch=tok.pad([{k:v for k,v in r.items() if k!='context_mask'} for r in rows],padding=True,pad_to_multiple_of=8,return_tensors='pt')
        batch={k:v.to('cuda') for k,v in batch.items()}
        if mode=='qa':
            L=batch['input_ids'].shape[1]
            mask=torch.tensor([r['context_mask']+[0]*(L-len(r['context_mask'])) for r in rows],device='cuda',dtype=torch.bool)
        else: mask=None
        return batch,mask
    def forward(indices):
        b,m=collate(indices)
        with torch.autocast(device_type='cuda',dtype=torch.float16):
            out=model(**b)
        if mode=='qa': return qa_score(out.start_logits,out.end_logits,m)
        return out.logits.float().reshape(-1)
    eval_bs=cfg['evalbatch']
    @torch.inference_mode()
    def predict(idx,desc):
        nonlocal eval_bs
        model.eval(); order=sorted(range(len(idx)),key=lambda j:len(enc[int(idx[j])]['input_ids']))
        out=np.zeros(len(idx),dtype=np.float32); k=0; start=time.monotonic()
        while k<len(order):
            b=order[k:k+eval_bs]
            try: out[b]=forward(idx[b]).cpu().numpy()
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache(); eval_bs//=2
                if eval_bs<1: raise
                print(tag,'reduced eval batch to',eval_bs,flush=True); continue
            k+=len(b)
            if k%max(1,eval_bs*80)<eval_bs or k==len(order):
                print(tag,desc,k,'/',len(order),'seconds',round(time.monotonic()-start,1),flush=True)
        if not np.isfinite(out).all(): raise RuntimeError(f'{tag}: non-finite inference scores')
        return out
    # Sample real-length rows to estimate how much time is needed for final inference.
    rng=np.random.default_rng(seed); probe=rng.choice(np.concatenate([v,te]),size=512,replace=False)
    predict(probe[:64],'warmup')
    t0=time.monotonic(); predict(probe,'speed probe'); elapsed=time.monotonic()-t0
    reserve=max(150.,elapsed/len(probe)*(len(v)+len(te))*1.35+80.)
    print(tag,'reserved inference seconds',round(reserve),flush=True)
    # AdamW keeps master weights in FP32; AMP is FP16 on T4, not BF16.
    optimizer=torch.optim.AdamW(params,lr=cfg['lr'],weight_decay=.01)
    scaler=torch.amp.GradScaler('cuda',init_scale=1024.)
    groups_per=cfg['groups']; labels=df.relevance.to_numpy(dtype=np.float32)
    done=0; updates=0; successful_queries=0; last_loss=0.; fixed=FIXED_BATCHES.get(tag)
    all_batches=[(epoch,b) for epoch in range(cfg['epochs']) for b in make_batches(df,a,enc,epoch,groups_per,seed)]
    # Each successful optimizer batch is made of whole queries, never sampled based on labels.
    def one_batch(groups):
        nonlocal last_loss,updates
        model.train(); optimizer.zero_grad(set_to_none=True)
        # Fixed weak dropout in frozen bottom layers is acceptable; gradients do not enter them.
        idx=np.concatenate(groups); y=torch.tensor(labels[idx],device='cuda')
        s=forward(idx); loss=rank_loss(s,y,[len(g) for g in groups])
        if not bool(torch.isfinite(loss)): raise RuntimeError('Non-finite training loss')
        scaler.scale(loss).backward(); scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(params,1.0)
        scale_before=scaler.get_scale(); scaler.step(optimizer); scaler.update()
        if scaler.get_scale()>=scale_before: updates+=1
        last_loss=float(loss.detach().cpu())
    start=time.monotonic()
    for epoch,groups in all_batches:
        if fixed is not None and done>=fixed: break
        if fixed is None and done>=12 and time.time()+reserve>=deadline: break
        if done>=12 and time.time()+30>=deadline: break
        # A short warmup followed by a conservative second-epoch decrease.
        warm=min(1.,(done+1)/30.)
        for pg in optimizer.param_groups: pg['lr']=cfg['lr']*warm*(.55 if epoch else 1.)
        try: one_batch(groups)
        except torch.cuda.OutOfMemoryError:
            optimizer.zero_grad(set_to_none=True); torch.cuda.empty_cache()
            if len(groups)==1: raise
            # Retry each intact query separately; never silently discard candidates.
            for group in groups: one_batch([group])
        done+=1; successful_queries+=len(groups)
        if done%50==0 or done==12:
            print(tag,'batches',done,'queries',successful_queries,'updates',updates,'loss',round(last_loss,4),'elapsed',round(time.monotonic()-start),'sec',flush=True)
    if updates==0: raise RuntimeError(f'{tag}: no successful optimizer update')
    # Record the actual schedule before inference. Save learned weights when inference completes.
    rev=model._amml_revision
    info={'tag':tag,'mode':mode,'model':MODEL_IDS[mode],'revision':rev,'config':cfg,'batches':done,'optimizer_updates':updates,'queries_seen':successful_queries,'eval_batch':eval_bs,'seed':seed,'torch':torch.__version__}
    atomic_json(info,OUT/f'fast_{tag}_run.json')
    # Save test first, so a completed fine-tuned submission remains available even if validation is interrupted.
    s_test=predict(te,'test'); submission(df,te,s_test,sample,OUT/f'submission_fast_{tag}.csv')
    s_val=predict(v,'validation'); val=ndcg(df,v,s_val)
    info.update(ndcg=val,eval_batch=eval_bs); atomic_json(info,OUT/f'fast_{tag}_run.json')
    tmp=OUT/f'fast_{tag}_scores.tmp.npz'; np.savez(tmp,valid=s_val,test=s_test); os.replace(tmp,OUT/f'fast_{tag}_scores.npz')
    atomic_csv(pd.DataFrame({'id':df.id.iloc[v].to_numpy(),'query_id':df.query_id.iloc[v].to_numpy(),'prediction':s_val,'relevance':df.relevance.iloc[v].astype(int).to_numpy()}),OUT/f'fast_{tag}_validation.csv')
    # Only learned parameter tensors, not the multi-GB base model.
    from safetensors.torch import save_file
    learned={k:p.detach().cpu().contiguous() for k,p in model.named_parameters() if p.requires_grad}
    save_file(learned,str(OUT/f'fast_{tag}_learned.safetensors'))
    print(tag,'NDCG@10',val,'complete',flush=True)


def raw_features(df,idx):
    sub=df.iloc[idx]; result=[]
    for q,t,title,miss,pi,pc,ac in zip(sub['query'],sub.text,sub.title_clean,sub.title_missing,sub.passage_index,sub.article_passage_count,sub.article_char_count):
        qs=keyset(q); ts=keyset(t); hs=keyset(title)
        def cov(x): return len(qs&x)/max(len(qs),1)
        def number(x):
            try: return max(0.,float(re.sub(r'[^\d.]','',str(x).replace(',', '')))) if pd.notna(x) else 0.
            except: return 0.
        first=words(q)[:2]; date=int('когда' in first or 'году' in words(q)[:4]); count=int('сколько' in first)
        numeric=bool(re.search(r'\d',t)); year=bool(re.search(r'\b(?:[12]\d{3})\b',t))
        result.append([cov(ts),cov(hs),cov(ts|hs),cov(keyset(t[:300])),np.log1p(len(t)),float(miss),np.log1p(number(pi)),np.log1p(number(pc)),np.log1p(number(ac)),float(pd.isna(pi)),float(pd.isna(pc)),float(pd.isna(ac)),date*year,count*numeric])
    return np.asarray(result,dtype=np.float64)

def meta_features(df,idx,raw,zs,anchor):
    sub=df.iloc[idx]; groups=group_positions(df,idx); feats=[]
    rawrel=np.column_stack([normalize(df,idx,raw[:,j]) for j in range(raw.shape[1])])
    feats.extend([rawrel,raw])
    arts=sub.article_id.to_numpy(); scoremat=np.column_stack(zs)
    # Relative neural scores and same-article support; IDs never used as numerical predictors.
    for sc in [*zs,anchor]:
        ranks=np.zeros(len(idx)); mx=np.zeros(len(idx)); mean=np.zeros(len(idx)); cnt=np.zeros(len(idx)); margin=np.zeros(len(idx))
        for g in groups:
            order=np.argsort(-sc[g],kind='stable'); ranks[g[order]]=1/(np.arange(len(g))+1)
            margin[g]=sc[g]-np.max(sc[g]);
            for j in g:
                other=g[(arts[g]==arts[j])&(g!=j)]
                if len(other): mx[j]=np.max(sc[other]); mean[j]=np.mean(sc[other]); cnt[j]=np.log1p(len(other))
        feats.extend([sc[:,None],ranks[:,None],mx[:,None],mean[:,None],cnt[:,None],margin[:,None]])
    if len(zs)>1: feats.append(np.std(scoremat,axis=1)[:,None])
    return np.column_stack(feats)

def residual_fit(df,idx,X,anchor,reg=.2):
    from scipy.optimize import minimize
    mean=X.mean(0); std=np.maximum(X.std(0),.1); X=np.clip((X-mean)/std,-5,5)
    sub=df.iloc[idx]; y=sub.relevance.to_numpy(); rows=[]; offsets=[]; ws=[]
    for g in group_positions(df,idx):
        pos=g[y[g]==1]; neg=g[y[g]==0]
        if not len(pos) or not len(neg): continue
        p=np.repeat(pos,len(neg)); n=np.tile(neg,len(pos)); rows.append(X[p]-X[n]); offsets.append(anchor[p]-anchor[n])
        order=np.argsort(-anchor[g],kind='stable'); r=np.zeros(len(idx),dtype=int); r[g[order]]=np.arange(len(g)); d=lambda k:np.where(k<10,1/np.log2(k+2),0.)
        w=np.abs(d(r[p])-d(r[n]))+.05; ws.append(w/w.sum())
    D=np.concatenate(rows); base=np.concatenate(offsets); w=np.concatenate(ws); w=w/w.sum()
    from scipy.special import expit
    def fun(beta):
        pred=base+D@beta
        return np.dot(w,np.logaddexp(0,-pred))+.5*reg*np.dot(beta,beta),D.T@(-w*expit(-pred))+reg*beta
    fit=minimize(fun,np.zeros(X.shape[1]),jac=True,method='L-BFGS-B',options={'maxiter':100})
    return fit.x,mean,std

def residual_predict(X,anchor,fit):
    beta,mean,std=fit; return anchor+np.clip((X-mean)/std,-5,5)@beta

def combine(df,v,te,sample):
    tags=[]; vs=[]; ts=[]; metrics={}
    for tag in ('rerank','qa'):
        path=OUT/f'fast_{tag}_scores.npz'
        if not path.exists(): continue
        z=np.load(path); sv=z['valid']; st=z['test']; assert len(sv)==len(v) and len(st)==len(te)
        tags.append(tag); vs.append(sv); ts.append(st); metrics[tag]=ndcg(df,v,sv)
    if not tags:
        # If final validation was interrupted, a trained independent test file is still usable.
        for tag in ('rerank','qa'):
            p=OUT/f'submission_fast_{tag}.csv'
            if p.exists():
                atomic_csv(pd.read_csv(p,dtype={'id':str}),OUT/'submission.csv')
                return {'selected':tag,'validation_unavailable':True}
        raise RuntimeError('No fine-tuned model produced a complete test file. Inspect logs.')
    vz=[normalize(df,v,s) for s in vs]; tz=[normalize(df,te,s) for s in ts]
    weights=[(1.,)] if len(tags)==1 else [(1.,0.),(.8,.2),(.65,.35),(.5,.5),(.35,.65),(.2,.8),(0.,1.)]
    options=[]
    for w in weights:
        vv=sum(x*k for x,k in zip(vz,w)); tt=sum(x*k for x,k in zip(tz,w)); options.append((ndcg(df,v,vv),w,vv,tt))
    base,w,av,at=max(options,key=lambda x:x[0]); av=2*av; at=2*at
    # Cross-fit both the blend weights and the regularized residual on held-out queries.
    # The final weights and residual are refit on all validation queries only after comparison.
    rawv=raw_features(df,v); rawt=raw_features(df,te)
    cv=np.zeros(len(v)); cv_blend=np.zeros(len(v))
    for tr,va in GroupKFold(5).split(rawv,groups=df.query_id.iloc[v]):
        fw=max(weights,key=lambda ww:ndcg(df,v[tr],sum(k*x[tr] for k,x in zip(ww,vz))))
        fold_anchor=2*sum(k*x for k,x in zip(fw,vz))
        Xfold=meta_features(df,v,rawv,vz,fold_anchor)
        fit=residual_fit(df,v[tr],Xfold[tr],fold_anchor[tr]); cv[va]=residual_predict(Xfold[va],fold_anchor[va],fit)
        cv_blend[va]=fold_anchor[va]
    cal_score=ndcg(df,v,cv); blend_cv_score=ndcg(df,v,cv_blend)
    X=meta_features(df,v,rawv,vz,av); T=meta_features(df,te,rawt,tz,at)
    fit=residual_fit(df,v,X,av); caltest=residual_predict(T,at,fit)
    submission(df,te,at,sample,OUT/'submission_fast_blend.csv')
    submission(df,te,caltest,sample,OUT/'submission_fast_calibrated.csv')
    use_cal=cal_score>blend_cv_score+.001
    chosen=caltest if use_cal else at; selected_val=cv if use_cal else cv_blend
    submission(df,te,chosen,sample,OUT/'submission.csv')
    atomic_csv(pd.DataFrame({'id':df.id.iloc[v].to_numpy(),'query_id':df.query_id.iloc[v].to_numpy(),'relevance':df.relevance.iloc[v].astype(int).to_numpy(),'prediction':selected_val,**{t:s for t,s in zip(tags,vs)}}),OUT/'fast_validation.csv')
    result={'individual_ndcg':metrics,'blend_ndcg':base,'blend_crossfit_ndcg':blend_cv_score,'weights':dict(zip(tags,w)),'calibrated_crossfit_ndcg':cal_score,'selected':'calibrated' if use_cal else 'blend','blend_grid':[{'ndcg':s,'weights':dict(zip(tags,ww))} for s,ww,_,_ in options]}
    atomic_json(result,OUT/'fast_metrics.json'); print(json.dumps(result,indent=2),flush=True); return result


def launch(deadline):
    import torch
    df,a,v,te,sample=prepare(); n=torch.cuda.device_count(); print('GPUs:',[torch.cuda.get_device_name(i) for i in range(n)],flush=True)
    assert n>=2,'Select GPU T4 x2, not one GPU; this notebook expects two devices.'
    # Both independent models use one GPU; the parent never allocates GPU model memory.
    jobs=[]
    for j,tag in enumerate(('rerank','qa')):
        env=os.environ.copy(); env['CUDA_VISIBLE_DEVICES']=str(j)
        p=subprocess.Popen([sys.executable,'-u',str(Path(__file__).resolve()),'--worker',tag,'--deadline',str(deadline-45)],env=env,stdout=open(OUT/f'fast_{tag}.log','w'),stderr=subprocess.STDOUT)
        jobs.append((tag,p)); print('Started',tag,'GPU',j,flush=True)
    positions={t:0 for t,_ in jobs}; previous=None; retried=False
    while True:
        for tag,p in jobs:
            with open(OUT/f'fast_{tag}.log','r',errors='replace') as f:
                f.seek(positions[tag]); s=f.read(); positions[tag]=f.tell()
            if s.strip(): print('['+tag+'] '+s[-4500:],flush=True)
        complete=tuple(tag for tag,_ in jobs if (OUT/f'fast_{tag}_scores.npz').exists())
        # Save primary file as soon as the first validated worker finishes; do not await the slower worker.
        if complete and complete!=previous:
            combine(df,v,te,sample); previous=complete
        # If the custom Jina runtime fails early, try a standard compact MiniLM model on the same GPU.
        for j,(tag,p) in enumerate(jobs):
            if tag=='rerank' and p.poll() not in (None,0) and not retried and not (OUT/'submission_fast_rerank.csv').exists() and time.time()+12*60<deadline:
                retried=True; env=os.environ.copy(); env['CUDA_VISIBLE_DEVICES']='0'
                q=subprocess.Popen([sys.executable,'-u',str(Path(__file__).resolve()),'--worker','rerank','--fallback','--deadline',str(deadline-45)],env=env,stdout=open(OUT/'fast_rerank.log','a'),stderr=subprocess.STDOUT)
                jobs[j]=(tag,q); print('Retry: MiniLM fallback after early reranker failure',flush=True)
        if all(p.poll() is not None for _,p in jobs): break
        if time.time()>=deadline-30:
            print('Budget cutoff: preserving completed files',flush=True)
            for _,p in jobs:
                if p.poll() is None:p.terminate()
            for _,p in jobs:
                try:p.wait(timeout=5)
                except subprocess.TimeoutExpired:p.kill()
            break
        time.sleep(5)
    result=combine(df,v,te,sample)
    infos={}
    for tag,p in jobs:
        path=OUT/f'fast_{tag}_run.json'
        if path.exists(): infos[tag]=json.loads(path.read_text())
    import platform,importlib.metadata
    packages={}
    for name in ['torch','transformers','accelerate','sentencepiece','einops','numpy','pandas','scipy','scikit-learn','safetensors','huggingface-hub','tokenizers']:
        try: packages[name]=importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError: pass
    atomic_json({'system':platform.platform(),'models':infos,'selection':result,'packages':packages,'worker_exit_codes':{tag:p.poll() for tag,p in jobs}},OUT/'fast_manifest.json')
    print('Finished:',OUT/'submission.csv',flush=True)

if __name__=='__main__':
    p=argparse.ArgumentParser(); p.add_argument('--worker',choices=['rerank','qa']); p.add_argument('--launch',action='store_true'); p.add_argument('--fallback',action='store_true'); p.add_argument('--deadline',type=float,default=None); a=p.parse_args()
    deadline=a.deadline or time.time()+35*60
    if a.worker: worker(a.worker,deadline,a.fallback)
    elif a.launch: launch(deadline)
    else: p.error('Specify --launch (or use the notebook)')


In [ ]:
%%writefile /kaggle/working/amml_repro_cache/solution_jina_refine.py
"""Jina refinement on competition data; requires saved fast Jina output."""
import os
os.environ.setdefault('USE_TF','0')
os.environ.setdefault('USE_FLAX','0')
os.environ.setdefault('TOKENIZERS_PARALLELISM','false')
os.environ.setdefault('OMP_NUM_THREADS','2')
os.environ.setdefault('MKL_NUM_THREADS','2')
os.environ.setdefault('OPENBLAS_NUM_THREADS','2')
import sys, argparse, time, json, math, re, html, unicodedata, random, traceback, subprocess
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, GroupKFold

ROOT=Path(os.environ.get('AMML_INPUT','/kaggle/input'))
OUT=Path(os.environ.get('AMML_OUTPUT','/kaggle/working'))
WORK=Path(os.environ.get('AMML_TEMP','/tmp/amml_jina_refine'))
SEED=731
MODEL_IDS={'rerank':'jinaai/jina-reranker-v2-base-multilingual',
           'qa':'timpal0l/mdeberta-v3-base-squad2',
           'fallback':'cross-encoder/mmarco-mMiniLMv2-L12-H384-v1'}
CONFIG={'rerank':dict(seed=7311,maxlen=288,top=6,lr=2e-5,epochs=2,groups=2,evalbatch=40),
        'qa':dict(seed=7312,maxlen=288,top=4,lr=1.5e-5,epochs=2,groups=2,evalbatch=32),
        'fallback':dict(seed=7313,maxlen=288,top=10,lr=2.5e-5,epochs=2,groups=3,evalbatch=64)}
# For exact reproduction, fill from fast_manifest.json, including selected mode/model/step count.
FIXED_BATCHES={}  # e.g. {'rerank': 800, 'qa': 600}; no time-adaptive training if populated
MODEL_REVISIONS={}  # fill with commit hashes from the manifest for a final fixed replay
MODEL_OVERRIDE={}  # e.g. {'rerank': 'fallback'} if the actual successful run used MiniLM
STOPWORDS=set('когда сколько кто какой какая какие какое каких каких как где чем что почему зачем куда откуда был была были было является называется это в во на по для при из с со о об у и а но же ли не до после за к от его ее или который которая'.split())

def atomic_csv(df,path):
    path=Path(path); tmp=path.with_suffix('.tmp'); df.to_csv(tmp,index=False); os.replace(tmp,path)

def atomic_json(obj,path):
    path=Path(path); tmp=path.with_suffix('.tmp'); tmp.write_text(json.dumps(obj,ensure_ascii=False,indent=2),encoding='utf-8'); os.replace(tmp,path)

def locate(stem,n):
    override=os.environ.get('AMML_'+stem.upper())
    paths=[Path(override)] if override else sorted(set(ROOT.rglob(stem+'.csv')) | set(ROOT.rglob(stem+' (*)'+'.csv')))
    for p in paths:
        try:
            df=pd.read_csv(p,dtype={'id':str,'query_id':str,'article_id':str,'document_id':str})
            if len(df)==n and ('relevance' in df)==(stem=='train'): return df,p
        except Exception: continue
    raise FileNotFoundError(f'Cannot find competition {stem}.csv with {n} rows; candidates: {paths}')

def repair(x):
    # Undo a reversible UTF-8 decoding error, only with a Cyrillic result.
    for codec in ('cp1251','latin1','cp1252'):
        try:
            y=x.encode(codec).decode('utf8')
            if y!=x and re.search('[А-Яа-яЁё]',y): return y
        except UnicodeError: pass
    return x

def clean(x):
    if pd.isna(x): return ''
    s=unicodedata.normalize('NFKC',html.unescape(repair(str(x))))
    s=s.replace('\u200b','').replace('\ufeff','')
    s=re.sub(r'</?(?:p|div|span|br|b|i|strong|em)(?:\s[^>]*)?/?>',' ',s,flags=re.I)
    return re.sub(r'\s+',' ',s).strip()

def clean_title(x):
    return re.sub(r'\s*[-–—|]\s*Википедия\s*$','',clean(x),flags=re.I).strip()

def words(x): return re.findall(r'[а-яёa-z0-9]+',str(x).lower().replace('ё','е'))

def keyset(x): return {w[:5] for w in words(x) if w not in STOPWORDS and len(w)>2}

def focus(q,t,maxchars=1500):
    if len(t)<=maxchars: return t
    # Retain the beginning and the most query-matching contiguous later window.
    # No label, external corpus, or answer lookup is used.
    qs=keyset(q)
    head=t[:420]
    width=maxchars-440
    starts=range(0,max(1,len(t)-width+1),max(160,width//3))
    opts=list(starts)+[max(0,len(t)-width)]
    def score(i):
        w=keyset(t[i:i+width]); return (len(w&qs),-i)
    best=max(opts,key=score)
    if best<=300: return t[:maxchars]
    return head+' … '+t[best:best+width]

def prepare():
    WORK.mkdir(parents=True,exist_ok=True); OUT.mkdir(parents=True,exist_ok=True)
    tr,p=locate('train',31530); te,p2=locate('test',13507); sample,p3=locate('sample_submission',13507)
    assert set(sample.id)==set(te.id) and sample.id.is_unique and tr.id.is_unique and te.id.is_unique
    assert not set(tr.query_id)&set(te.query_id)
    df=pd.concat([tr,te],ignore_index=True); n=len(tr)
    df['query']=df['query'].map(clean); df['text']=df['text'].map(clean)
    df['title_clean']=df['title'].map(clean_title)
    df['title_missing']=df.title_clean.eq('').astype(int)
    known=df.loc[df.title_clean.ne('')].groupby('article_id').title_clean.agg(['first','nunique'])
    known=known.loc[known['nunique'].eq(1),'first']
    mask=df.title_clean.eq('')
    df.loc[mask,'title_clean']=df.loc[mask,'article_id'].map(known).fillna('')
    print('Restored titles',int((mask&df.title_clean.ne('')).sum()),flush=True)
    df['doc_view']=[(title[:250]+'\n' if title else '')+focus(q,t,2400) for q,t,title in zip(df['query'],df.text,df.title_clean)]
    a,v=next(GroupShuffleSplit(n_splits=1,test_size=.15,random_state=SEED).split(tr,groups=tr.query_id))
    assert not set(df.query_id.iloc[a])&set(df.query_id.iloc[v]); assert len(a)+len(v)==n
    test=np.arange(n,len(df)); df.to_pickle(WORK/'data.pkl'); sample[['id']].to_csv(WORK/'sample.csv',index=False)
    np.savez(WORK/'split.npz',train=a,valid=v,test=test)
    print('Data',p,p2,p3,'groups',df.query_id.iloc[a].nunique(),df.query_id.iloc[v].nunique(),flush=True)
    return df,a,v,test,sample[['id']]

def get_data():
    df=pd.read_pickle(WORK/'data.pkl'); z=np.load(WORK/'split.npz'); return df,z['train'],z['valid'],z['test'],pd.read_csv(WORK/'sample.csv',dtype={'id':str})

def group_positions(df,idx):
    sub=df.iloc[idx]; return [np.asarray(x,dtype=int) for x in sub.groupby('query_id',sort=False).indices.values()]

def ndcg_values(df,idx,scores):
    s=np.asarray(scores,float); assert s.shape==(len(idx),) and np.isfinite(s).all()
    sub=df.iloc[idx]; y=sub.relevance.to_numpy(); ids=sub.id.astype(str).to_numpy(); d=1/np.log2(np.arange(2,12)); vals=[]
    for g in group_positions(df,idx):
        order=np.lexsort((ids[g],-s[g])); k=min(len(g),10); ideal=float(d[:min(int(y[g].sum()),10)].sum())
        vals.append(float(np.dot(y[g[order[:k]]],d[:k])/ideal) if ideal else 0.)
    return np.asarray(vals)

def ndcg(df,idx,scores): return float(ndcg_values(df,idx,scores).mean())

def normalize(df,idx,scores):
    s=np.asarray(scores,dtype=np.float64); o=np.zeros(len(s))
    for g in group_positions(df,idx):
        x=s[g]; o[g]=np.clip((x-x.mean())/max(x.std(),1e-5),-4,4)
    return o

def submission(df,idx,scores,sample,path):
    assert np.isfinite(scores).all() and len(scores)==len(idx)
    out=sample.merge(pd.DataFrame({'id':df.id.iloc[idx].to_numpy(),'prediction':scores}),on='id',how='left',validate='one_to_one')
    assert len(out)==13507 and not out.prediction.isna().any(); atomic_csv(out,path)

def make_batches(df,a,enc,epoch,groups_per,seed):
    groups=[a[g] for g in group_positions(df,a)]
    rng=np.random.default_rng(seed+epoch); rng.shuffle(groups)
    # Length bucketing within randomized small pools: reduce padding without sorting by labels.
    batches=[]
    for j in range(0,len(groups),64):
        chunk=sorted(groups[j:j+64],key=lambda g:max(len(enc[int(i)]['input_ids']) for i in g))
        bag=[chunk[k:k+groups_per] for k in range(0,len(chunk),groups_per)]
        rng.shuffle(bag); batches.extend(bag)
    return batches

def rank_loss(scores,labels,lengths):
    import torch
    import torch.nn.functional as F
    losses=[]; offset=0
    for n in lengths:
        s=scores[offset:offset+n].float(); y=labels[offset:offset+n].float(); offset+=n
        bce=F.binary_cross_entropy_with_logits(s,y,pos_weight=s.new_tensor(2.0))
        # All judged candidates of a query are present in the batch.
        with torch.no_grad():
            ranks=torch.argsort(torch.argsort(s,descending=True,stable=True),stable=True)
            disc=torch.where(ranks<10,1/torch.log2(ranks.float()+2),0.)
            match=(y[:,None]>y[None,:]).float()
            ideal=(1/torch.log2(torch.arange(n,device=s.device).float()+2))*((torch.arange(n,device=s.device)<y.sum())&(torch.arange(n,device=s.device)<10))
            w=((disc[:,None]-disc[None,:]).abs()/ideal.sum().clamp_min(1e-6)+0.02)*match
        pair=(F.softplus(s[None,:]-s[:,None])*w).sum()/w.sum().clamp_min(1e-6)
        listwise=-(F.log_softmax(s,dim=0)*y).sum()/y.sum().clamp_min(1)
        losses.append(.44*bce+.46*pair+.10*listwise)
    return torch.stack(losses).mean()

def qa_score(start,end,mask):
    """Best valid contiguous answer span vs CLS no-answer, max width 32 tokens."""
    import torch
    import torch.nn.functional as F
    start=start.float(); end=end.float(); mask=mask.bool(); L=start.shape[1]
    # Restrict start/end to actual document tokens, not padding/question/special tokens.
    ends=F.pad(end.masked_fill(~mask,-1e4),(0,31),value=-1e4).unfold(1,32,1)
    pairs=start.masked_fill(~mask,-1e4).unsqueeze(-1)+ends
    best=pairs.flatten(1).max(dim=1).values
    null=start[:,0]+end[:,0]
    return (best-null)/2

REV='9cfeff2df7d40d1b78e75e5e9cebec92a99813c9'
NAME='jinaai/jina-reranker-v2-base-multilingual'
CFG={'a':dict(seed=9411,maxlen=384,lr=6e-6,epochs=2,groups=1,evalbatch=32,resume=True),
     'b':dict(seed=9412,maxlen=320,lr=1.7e-5,epochs=2,groups=2,evalbatch=32,resume=False)}

def unique(name,required=True):
    paths=sorted(ROOT.rglob(name))
    if len(paths)>1: raise ValueError(f'Attach only one copy of {name}: {paths}')
    if not paths:
        if required: raise FileNotFoundError(name)
        return None
    return paths[0]

def read_scores(path,df,idx,check_labels=False):
    z=pd.read_csv(path,dtype={'id':str,'query_id':str}); sub=df.iloc[idx]
    assert z.id.is_unique and set(z.id)==set(sub.id),f'IDs mismatch: {path}'
    z=z.set_index('id').loc[sub.id]
    if 'query_id' in z: assert np.array_equal(z.query_id,sub.query_id)
    if check_labels and 'relevance' in z: assert np.array_equal(z.relevance,sub.relevance)
    x=z.prediction.to_numpy(float); assert np.isfinite(x).all(); return x

def old_scores(df,v,te):
    a=read_scores(unique('fast_rerank_validation.csv'),df,v,True)
    b=read_scores(unique('submission_fast_rerank.csv'),df,te)
    assert abs(ndcg(df,v,a)-.9494168923947727)<1e-6, 'Expected original Jina validation output'
    return a,b

def worker_refine(tag,deadline):
    import torch
    from transformers import AutoTokenizer,AutoModelForSequenceClassification
    from safetensors.torch import load_file,save_file
    cfg=CFG[tag]; seed=cfg['seed']; random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.set_num_threads(2); torch.cuda.set_device(0)
    df,a,v,te,sample=get_data()
    tok=AutoTokenizer.from_pretrained(NAME,revision=REV,use_fast=True); tok.padding_side='right'
    model=AutoModelForSequenceClassification.from_pretrained(NAME,revision=REV,trust_remote_code=True,use_flash_attn=False,torch_dtype=torch.float32)
    if cfg['resume']:
        old=load_file(str(unique('fast_rerank_learned.safetensors')))
        named=dict(model.named_parameters())
        assert old and all(k in named and named[k].shape==x.shape for k,x in old.items())
        with torch.no_grad():
            for k,x in old.items(): named[k].copy_(x)
        print('Restored',len(old),'learned tensors',flush=True); del old
    for p in model.parameters():p.requires_grad_(False)
    layers=getattr(model.base_model.encoder,'layers',None)
    if layers is None: layers=model.base_model.encoder.layer
    for layer in layers:
        for p in layer.parameters():p.requires_grad_(True)
    for p in model.classifier.parameters():p.requires_grad_(True)
    model.to('cuda'); params=[p for p in model.parameters() if p.requires_grad]
    print(tag,'trainable',sum(p.numel() for p in params),'layers',len(layers),flush=True)
    enc=[]
    for start in range(0,len(df),512):
        sub=df.iloc[start:start+512]
        # Longer view; avoid truncating the question.
        x=tok(sub['query'].tolist(),sub.doc_view.tolist(),truncation='only_second',max_length=cfg['maxlen'],padding=False)
        enc.extend([{k:x[k][j] for k in ['input_ids','attention_mask','token_type_ids'] if k in x} for j in range(len(sub))])
    def forward(indices):
        batch=tok.pad([enc[int(i)] for i in indices],padding=True,pad_to_multiple_of=8,return_tensors='pt')
        with torch.autocast('cuda',dtype=torch.float16): out=model(**{k:x.to('cuda') for k,x in batch.items()})
        return out.logits.float().reshape(-1)
    eval_bs=cfg['evalbatch']
    @torch.inference_mode()
    def predict(idx):
        nonlocal eval_bs
        model.eval(); order=np.argsort([len(enc[int(i)]['input_ids']) for i in idx],kind='stable'); out=np.empty(len(idx),dtype=np.float32); k=0
        while k<len(idx):
            j=order[k:k+eval_bs]
            try: out[j]=forward(idx[j]).cpu().numpy()
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache(); eval_bs//=2
                if eval_bs<1:raise
                continue
            k+=len(j)
        assert np.isfinite(out).all(); return out
    probe=np.random.default_rng(seed).choice(np.concatenate([v,te]),512,replace=False)
    predict(probe[:64]); t=time.monotonic();predict(probe); per_row=(time.monotonic()-t)/len(probe)
    reserve=max(150.,1.6*per_row*(len(v)+len(te))+75.)
    print('Inference reserve seconds',reserve,flush=True)
    opt=torch.optim.AdamW(params,lr=cfg['lr'],weight_decay=.01); scaler=torch.amp.GradScaler('cuda',init_scale=1024.)
    labels=df.relevance.to_numpy(np.float32); best=None; best_score=-1.; best_val=None; updates=0; history=[]; examples=0; total_batches=0
    # Full-query loss; retry with one intact query if a two-query batch runs out of memory.
    def step(groups):
        nonlocal updates
        model.train(); opt.zero_grad(set_to_none=True)
        ii=np.concatenate(groups); y=torch.tensor(labels[ii],device='cuda'); loss=rank_loss(forward(ii),y,[len(g) for g in groups]); assert bool(torch.isfinite(loss))
        scaler.scale(loss).backward();scaler.unscale_(opt);torch.nn.utils.clip_grad_norm_(params,1.); old=scaler.get_scale();scaler.step(opt);scaler.update(); updates+=int(scaler.get_scale()>=old)
    # Inference after each completed (or budget-limited partial) epoch. Keep best checkpoint in host RAM.
    for epoch in range(cfg['epochs']):
        count=0; stop=False
        for groups in make_batches(df,a,enc,epoch,cfg['groups'],seed):
            if tag=='a' and epoch==1 and count>=2293: stop=True;break  # Recorded 2293 second-epoch queries (A)
            if time.time()+reserve>=deadline: stop=True;break
            for pg in opt.param_groups:pg['lr']=cfg['lr']*min(1.,(total_batches+1)/40.)*(.5 if epoch else 1.)
            try: step(groups)
            except torch.cuda.OutOfMemoryError:
                opt.zero_grad(set_to_none=True);torch.cuda.empty_cache()
                if len(groups)==1: raise
                for g in groups: step([g])
            count+=len(groups);examples+=len(groups);total_batches+=1
            if total_batches%100==0:print(tag,'epoch',epoch+1,'queries',count,'updates',updates,flush=True)
        if count==0:break
        sv=predict(v); score=ndcg(df,v,sv);history.append(dict(epoch=epoch+1,queries=count,updates=updates,ndcg=score)); print(tag,history[-1],flush=True)
        if score>best_score:
            best_score=score;best_val=sv.copy();best={k:p.detach().cpu().clone() for k,p in model.named_parameters() if p.requires_grad}; best_updates=updates
        if stop or time.time()+reserve>=deadline:break
    if best is None or updates==0: raise RuntimeError('No trained checkpoint; anchor is retained')
    with torch.no_grad():
        for k,p in model.named_parameters():
            if k in best:p.copy_(best[k])
    st=predict(te)
    submission(df,te,st,sample,OUT/f'submission_jina_{tag}.csv')
    atomic_csv(pd.DataFrame({'id':df.id.iloc[v].to_numpy(),'query_id':df.query_id.iloc[v].to_numpy(),'prediction':best_val,'relevance':labels[v].astype(int)}),OUT/f'jina_{tag}_validation.csv')
    atomic_json(dict(config=cfg,revision=REV,history=history,best_ndcg=best_score,best_updates=best_updates,updates=updates,queries_seen=examples),OUT/f'jina_{tag}_run.json')
    tmp=OUT/f'jina_{tag}_scores.tmp.npz';np.savez(tmp,valid=best_val,test=st);os.replace(tmp,OUT/f'jina_{tag}_scores.npz')
    # Save FP32 learned weights only; the original frozen embeddings come from REV.
    save_file(best,str(OUT/f'jina_{tag}_learned.safetensors'))
    print(tag,'DONE',best_score,flush=True)

def combine_refine(df,v,te,sample):
    av,at=old_scores(df,v,te); names=['old'];vs=[normalize(df,v,av)];ts=[normalize(df,te,at)]
    for tag in CFG:
        p=OUT/f'jina_{tag}_scores.npz'
        if p.exists():
            z=np.load(p);assert len(z['valid'])==len(v) and len(z['test'])==len(te)
            names.append(tag);vs.append(normalize(df,v,z['valid']));ts.append(normalize(df,te,z['test']))
    # Small, pre-specified grid. No leaderboard-driven weights.
    opts=[np.eye(len(names))[i] for i in range(len(names))]
    for j in range(1,len(names)):
        for w in (.25,.5,.75):
            x=np.zeros(len(names));x[0]=1-w;x[j]=w;opts.append(x)
    if len(names)==3:opts.extend([np.array(x) for x in [(1/3,1/3,1/3),(.5,.25,.25),(0,.5,.5)]])
    options=[]; baseline_vals=ndcg_values(df,v,vs[0])
    for w in opts:
        sv=sum(k*x for k,x in zip(w,vs));st=sum(k*x for k,x in zip(w,ts)); vals=ndcg_values(df,v,sv);delta=vals-baseline_vals; se=delta.std(ddof=1)/np.sqrt(len(delta))
        options.append(dict(w=w,sv=sv,st=st,ndcg=vals.mean(),delta=delta.mean(),se=se,criterion=delta.mean()-.5*se))
    new=max(options[1:],key=lambda x:x['criterion']) if len(options)>1 else options[0]
    selected=options[0];  # Original successful selection was the old Jina reranker
    submission(df,te,selected['st'],sample,OUT/'submission.csv');submission(df,te,new['st'],sample,OUT/'submission_jina_candidate.csv')
    atomic_csv(pd.DataFrame({'id':df.id.iloc[v].to_numpy(),'query_id':df.query_id.iloc[v].to_numpy(),'relevance':df.relevance.iloc[v].astype(int).to_numpy(),'prediction':selected['sv'],**{n:x for n,x in zip(names,vs)}}),OUT/'jina_refine_validation.csv')
    def report(x):return dict(weights=dict(zip(names,map(float,x['w']))),ndcg=float(x['ndcg']),delta=float(x['delta']),paired_se=float(x['se']),criterion=float(x['criterion']))
    result=dict(individual={n:ndcg(df,v,x) for n,x in zip(names,vs)},selected=report(selected),candidate=report(new),grid=[report(x) for x in options])
    # Optional independent Qwen combination: fixed 50/50, evaluated ONLY on common unseen queries.
    qp=unique('submission_v6_selected.csv',False);qv=unique('v6_validation.csv',False)
    if qp is not None and qv is not None:
        qtest=read_scores(qp,df,te); mix=.5*normalize(df,te,qtest)+.5*selected['st']
        submission(df,te,mix,sample,OUT/'submission_jina_qwen_50.csv')
        q=pd.read_csv(qv,dtype={'id':str});assert q.id.is_unique
        common=np.flatnonzero(df.id.iloc[v].isin(q.id).to_numpy())
        if len(common):
            allcounts=df.iloc[v].groupby('query_id').size(); commoncounts=df.iloc[v[common]].groupby('query_id').size();assert (commoncounts==allcounts.loc[commoncounts.index]).all()
            qscore=q.set_index('id').loc[df.id.iloc[v[common]],'prediction'].to_numpy();jscore=selected['sv'][common];qz=normalize(df,v[common],qscore);jz=normalize(df,v[common],jscore)
            result['qwen_fixed_50_common_holdout']=dict(queries=len(commoncounts),jina=ndcg(df,v[common],jz),qwen=ndcg(df,v[common],qz),blend=ndcg(df,v[common],.5*(jz+qz)))
    atomic_json(result,OUT/'jina_refine_metrics.json');print(json.dumps(result,indent=2),flush=True)
    return result

def launch_refine(deadline,cpu=False):
    df,a,v,te,sample=prepare(); old_scores(df,v,te); combine_refine(df,v,te,sample)
    if cpu:return
    unique('fast_rerank_learned.safetensors')
    import torch
    assert torch.cuda.device_count()>=2,'Select T4 x2'
    jobs=[];positions={}
    for gpu,tag in enumerate(CFG):
        env=os.environ.copy();env['CUDA_VISIBLE_DEVICES']=str(gpu)
        log=OUT/f'jina_{tag}.log'
        with open(log,'w') as f:p=subprocess.Popen([sys.executable,'-u',__file__,'--worker',tag,'--deadline',str(deadline-45)],env=env,stdout=f,stderr=subprocess.STDOUT)
        jobs.append((tag,p));positions[tag]=0
    previous=()
    while True:
        for tag,p in jobs:
            with open(OUT/f'jina_{tag}.log',errors='replace') as f:f.seek(positions[tag]);s=f.read();positions[tag]=f.tell()
            if s.strip():print(f'[{tag}] {s[-4000:]}',flush=True)
        complete=tuple(t for t,_ in jobs if (OUT/f'jina_{t}_scores.npz').exists())
        if complete!=previous:combine_refine(df,v,te,sample);previous=complete
        if all(p.poll() is not None for _,p in jobs):break
        if time.time()>deadline-20:
            for _,p in jobs:
                if p.poll() is None:p.terminate()
            for _,p in jobs:
                try:p.wait(timeout=5)
                except subprocess.TimeoutExpired:p.kill();p.wait()
            break
        time.sleep(5)
    combine_refine(df,v,te,sample)
    print('Worker exit codes',[(t,p.poll()) for t,p in jobs],flush=True)

if __name__=='__main__':
    p=argparse.ArgumentParser();p.add_argument('--worker',choices=list(CFG));p.add_argument('--deadline',type=float,default=None);p.add_argument('--cpu',action='store_true');args=p.parse_args(); deadline=args.deadline or time.time()+25*60
    if args.worker:worker_refine(args.worker,deadline)
    else:launch_refine(deadline,args.cpu)


In [ ]:
%%writefile /kaggle/working/amml_repro_cache/solution_final_blend.py
"""AMML: conservative score fusion + optional multi-view inference of trained Jina.
No new training or external corpus in this stage; reproduce fast/refine/v5/v6 training
for final repository. Preserve the previous submitted anchor exactly.
"""
import os
for k,v in {'USE_TF':'0','USE_FLAX':'0','TOKENIZERS_PARALLELISM':'false','OMP_NUM_THREADS':'2','MKL_NUM_THREADS':'2','OPENBLAS_NUM_THREADS':'2'}.items(): os.environ.setdefault(k,v)
import argparse, json, math, sys, time, subprocess, re, html, unicodedata
from pathlib import Path
import numpy as np
import pandas as pd
ROOT=Path(os.environ.get('AMML_INPUT','/kaggle/input'))
OUT=Path(os.environ.get('AMML_OUTPUT','/kaggle/working'))
WORK=Path(os.environ.get('AMML_TEMP','/tmp/amml_final_blend'))
NAME='jinaai/jina-reranker-v2-base-multilingual'
REV='9cfeff2df7d40d1b78e75e5e9cebec92a99813c9'
VIEWS={'body':dict(maxlen=384,weights='fast_rerank_learned.safetensors'), 'long':dict(maxlen=512,weights='jina_b_learned.safetensors')}

def unique(name):
    p=sorted(ROOT.rglob(name)); assert len(p)==1,f'Expected one {name}; got {p}'; return p[0]

def competition():
    for p in sorted(ROOT.rglob('train.csv')):
        if not (p.parent/'sample_submission.csv').exists() or not (p.parent/'test.csv').exists(): continue
        a=pd.read_csv(p,dtype={'id':str,'query_id':str}); b=pd.read_csv(p.parent/'test.csv',dtype={'id':str,'query_id':str}); s=pd.read_csv(p.parent/'sample_submission.csv',dtype={'id':str})
        if len(a)==31530 and len(b)==13507 and 'relevance' in a and 'query' in b:
            assert a.id.is_unique and b.id.is_unique and s.id.is_unique and set(s.id)==set(b.id)
            assert not set(a.query_id)&set(b.query_id)
            return a,b,s[['id']]
    raise FileNotFoundError('Attach original competition data')

def table(p):
    x=pd.read_csv(p,dtype={'id':str,'query_id':str}); assert x.id.is_unique, p; return x.set_index('id')

def align(x,df,col='prediction'):
    assert set(x.index)==set(df.id), f'ID mismatch for {col}, {len(x)} versus {len(df)}'
    p=x.loc[df.id,col].to_numpy(float); assert np.isfinite(p).all(); return p

def groups(df): return list(df.groupby('query_id',sort=False).indices.values())

def z(df,p):
    p=np.asarray(p,dtype=float); o=np.empty(len(p))
    for g in groups(df):
        q=p[g]; o[g]=np.clip((q-q.mean())/max(q.std(),1e-5),-4,4)
    return o

def per_query(df,p):
    p=np.asarray(p,float); y=df.relevance.to_numpy(float); ids=df.id.to_numpy(str); d=1/np.log2(np.arange(2,12)); vals=[]
    assert len(p)==len(y) and np.isfinite(p).all()
    for g in groups(df):
        idx=np.lexsort((ids[g],-p[g])); k=min(10,len(g)); ideal=d[:min(10,int(y[g].sum()))].sum()
        vals.append(float(np.dot(y[g[idx[:k]]],d[:k])/ideal) if ideal else 0.)
    return np.array(vals)

def atomic_csv(d,path):
    p=Path(path); tmp=p.with_suffix('.tmp'); d.to_csv(tmp,index=False); os.replace(tmp,p)

def save(df,sample,p,path):
    assert len(p)==len(df) and np.isfinite(p).all()
    d=sample.merge(pd.DataFrame({'id':df.id,'prediction':p}),on='id',validate='one_to_one'); assert len(d)==13507
    atomic_csv(d,OUT/path)

def validate_subset(train,t):
    v=train[train.id.isin(t.index)].reset_index(drop=True)
    assert set(t.index)==set(v.id)
    assert len(v)==len(train[train.query_id.isin(v.query_id)]),'Partial query groups'
    if 'relevance' in t: assert np.array_equal(t.loc[v.id,'relevance'].to_numpy(),v.relevance.to_numpy())
    return v

def inputs():
    tr,te,sample=competition()
    oldv=table(unique('fast_rerank_validation.csv')); v=validate_subset(tr,oldv)
    ov=align(oldv,v); assert abs(per_query(v,ov).mean()-.9494168923947727)<1e-6,'Unexpected fast checkpoint'
    ref=unique('jina_refine_validation.csv').parent
    rv=table(ref/'jina_refine_validation.csv'); sv=align(rv,v)
    st=align(table(ref/'submission.csv'),te)
    qv=table(unique('v6_validation.csv')); qdf=validate_subset(tr,qv)
    qt=align(table(unique('submission_v6_selected.csv')),te); qzt=z(te,qt)
    anchor=align(table(ref/'submission_jina_qwen_50.csv'),te)
    assert np.allclose(anchor,.5*qzt+.5*st,rtol=1e-6,atol=1e-7),'Anchor does not match previous 50/50 recipe'
    common=v.query_id.isin(qdf.query_id).to_numpy(); excl=~common
    vc=v.loc[common].reset_index(drop=True)
    qcommon=z(vc,qv.loc[vc.id,'prediction'].to_numpy(float))
    anchor_v=.5*qcommon+.5*sv[common]
    family={'old':(z(v,ov),z(te,align(table(unique('submission_fast_rerank.csv')),te)))}
    for t in ('a','b'):
        vp=ref/f'jina_{t}_validation.csv'; tp=ref/f'submission_jina_{t}.csv'
        if vp.exists() and tp.exists(): family[t]=(z(v,align(table(vp),v)),z(te,align(table(tp),te)))
    assert len(family)>=2,'Missing saved refined Jina predictions'
    print('Jina validation groups:',v.query_id.nunique(),'common unseen with Qwen:',vc.query_id.nunique(),'Jina-only tuning:',v.loc[excl,'query_id'].nunique(),flush=True)
    assert vc.query_id.nunique()>=30 and v.loc[excl,'query_id'].nunique()>=100
    return tr,te,sample,v,common,excl,qzt,anchor,anchor_v,family,sv,st

def compare(df,base,cand):
    b=per_query(df,base); c=per_query(df,cand); delta=c-b; se=delta.std(ddof=1)/math.sqrt(len(delta))
    return dict(ndcg=float(c.mean()),delta=float(delta.mean()),paired_se=float(se),criterion=float(delta.mean()-.5*se))

def combine(data):
    tr,te,sample,v,common,excl,qzt,anchor,anchor_v,family,sv,st=data
    family=dict(family)
    for name in VIEWS:
        p=OUT/f'view_{name}.npz'
        if p.exists():
            a=np.load(p); family[name]=(z(v,a['valid']),z(te,a['test']))
    save(te,sample,anchor,'submission_anchor_0955450.csv')
    # Fixed prior: retain half Qwen, then choose a conservative Jina-only update.
    # Choose the family component on Jina-only held-out queries, NOT the common set.
    # Prior holdout was already used for previous experiments, so this is diagnostic, not pristine CV.
    val=v.loc[excl].reset_index(drop=True); vc=v.loc[common].reset_index(drop=True)
    names=list(family); pool=[('equal_family',np.full(len(names),1/len(names)))]
    # A modest grid of candidates, each anchored to the previously selected Jina scores.
    for n in names:
        if n!='old':
            w=np.zeros(len(names));w[names.index(n)]=1.;pool.append((n,w))
    scores=[]
    for name,w in pool:
        fv=sum(k*family[n][0] for n,k in zip(names,w)); ft=sum(k*family[n][1] for n,k in zip(names,w))
        for amount in (.25,.5):
            pv=(1-amount)*sv+amount*fv; pt=(1-amount)*st+amount*ft
            m=compare(val,sv[excl],pv[excl]); scores.append((m['criterion'],name,w,amount,pv,pt,m))
    _,name,w,amount,pv,pt,m=max(scores,key=lambda x:x[0])
    test=.5*qzt+.5*pt
    cand_v=anchor_v+.5*(pv[common]-sv[common])
    check=compare(vc,anchor_v,cand_v)
    # Do not tune a second grid on the small common set. One conservative gate.
    accept=check['delta']>max(.0005,check['paired_se']) and m['criterion']>0
    save(te,sample,test,'submission_final_candidate.csv')
    save(te,sample,test if accept else anchor,'submission.csv')
    # A single predeclared alternative: equal-weight family, 50/50 with Qwen.
    fixed_v=sum(p[0] for p in family.values())/len(family);fixed_t=sum(p[1] for p in family.values())/len(family)
    fixed_test=.5*qzt+.5*fixed_t
    fixed_check=compare(vc,anchor_v,anchor_v+.5*(fixed_v[common]-sv[common]))
    save(te,sample,fixed_test,'submission_final_equal.csv')
    info=dict(anchor_common_ndcg=float(per_query(vc,anchor_v).mean()),common_queries=int(vc.query_id.nunique()),family_tune_queries=int(val.query_id.nunique()),individual_jina_exclusive={n:float(per_query(val,p[0][excl]).mean()) for n,p in family.items()},candidate=dict(family=name,weights=dict(zip(names,map(float,w))),new_family_fraction=amount,jina_exclusive=m,common=check),equal_family_common=fixed_check,selected='candidate' if accept else 'anchor',warning='Reused holdouts, not independent evaluation. Public scores not used for tuning. Previous anchor chosen by user-reported result.')
    (OUT/'final_blend_metrics.json').write_text(json.dumps(info,indent=2),encoding='utf-8');print(json.dumps(info,indent=2),flush=True)
    return info

def repair(x):
    for c in ('cp1251','latin1','cp1252'):
        try:
            y=x.encode(c).decode('utf8')
            if y!=x and re.search('[А-Яа-яЁё]',y):return y
        except UnicodeError:pass
    return x

def clean(x):
    if pd.isna(x):return ''
    s=unicodedata.normalize('NFKC',html.unescape(repair(str(x))))
    s=s.replace('\u200b','').replace('\ufeff','');s=re.sub(r'</?(?:p|div|span|br|b|i|strong|em)(?:\s[^>]*)?/?>',' ',s,flags=re.I)
    return re.sub(r'\s+',' ',s).strip()

def prepare_views(data):
    tr,te,_,v,*_=data
    # Use only observed text/metadata, including titles from the supplied corpus.
    df=pd.concat([tr,te],ignore_index=True);df['title_clean']=df.title.map(clean).str.replace(r'\s*[-–—|]\s*Википедия\s*$','',regex=True,case=False)
    known=df.loc[df.title_clean.ne('')].groupby('article_id').title_clean.agg(['first','nunique']);known=known.loc[known['nunique']==1,'first']
    mask=df.title_clean.eq('');df.loc[mask,'title_clean']=df.loc[mask,'article_id'].map(known).fillna('')
    x=df.set_index('id').loc[list(v.id)+list(te.id),['query','text','title_clean']].reset_index();x['query']=x['query'].map(clean);x['text']=x['text'].map(clean)
    WORK.mkdir(parents=True,exist_ok=True);x.to_pickle(WORK/'views.pkl')
    (WORK/'counts.json').write_text(json.dumps({'valid':len(v),'test':len(te)}))

def worker(name,deadline):
    import torch
    from transformers import AutoTokenizer,AutoModelForSequenceClassification
    from safetensors.torch import load_file
    cfg=VIEWS[name];torch.set_num_threads(2);torch.cuda.set_device(0)
    p=unique(cfg['weights']); tok=AutoTokenizer.from_pretrained(NAME,revision=REV,use_fast=True);tok.padding_side='right'
    model=AutoModelForSequenceClassification.from_pretrained(NAME,revision=REV,trust_remote_code=True,use_flash_attn=False,torch_dtype=torch.float32)
    old=load_file(str(p)); named=dict(model.named_parameters());assert old and all(k in named and named[k].shape==a.shape for k,a in old.items())
    with torch.no_grad():
        for k,a in old.items():named[k].copy_(a)
    del old;model.eval().to('cuda');model.requires_grad_(False)
    df=pd.read_pickle(WORK/'views.pkl'); count=json.loads((WORK/'counts.json').read_text())['valid']
    if name=='body': docs=df.text.tolist()
    else: docs=[(h[:250]+'\n' if h else '')+t for h,t in zip(df.title_clean,df.text)]
    enc=[]
    for i in range(0,len(df),512):
        if time.time()>deadline:raise TimeoutError('View preparation exceeded budget')
        x=tok(df['query'].iloc[i:i+512].tolist(),docs[i:i+512],truncation=True,max_length=cfg['maxlen'],padding=False)
        enc.extend([{k:x[k][j] for k in ('input_ids','attention_mask','token_type_ids') if k in x} for j in range(len(x['input_ids']))])
    order=np.argsort([len(x['input_ids']) for x in enc],kind='stable');p=np.empty(len(df),np.float32);bs=32;k=0;start=time.time()
    with torch.inference_mode():
        while k<len(order):
            if time.time()>deadline:raise TimeoutError('Inference time limit: retained CPU candidate')
            ii=order[k:k+bs];batch=tok.pad([enc[int(j)] for j in ii],padding=True,pad_to_multiple_of=8,return_tensors='pt')
            try:
                with torch.autocast('cuda',dtype=torch.float16): out=model(**{key:x.to('cuda') for key,x in batch.items()}).logits.float().reshape(-1)
                p[ii]=out.cpu().numpy();k+=len(ii)
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache();bs//=2
                if bs<1:raise
                continue
            if k%(bs*50)<bs or k==len(order): print(name,k,'/',len(order),'sec',round(time.time()-start),flush=True)
    assert np.isfinite(p).all();tmp=OUT/f'view_{name}.tmp.npz';np.savez(tmp,valid=p[:count],test=p[count:]);os.replace(tmp,OUT/f'view_{name}.npz')

def main(gpu,deadline):
    OUT.mkdir(parents=True,exist_ok=True); data=inputs(); combine(data)
    if not gpu:return
    import torch
    assert torch.cuda.device_count()>=2,'Select T4 x2 or CPU mode'
    prepare_views(data);jobs=[]
    for i,name in enumerate(VIEWS):
        try:unique(VIEWS[name]['weights'])
        except AssertionError as e:print('Skipping',name,e);continue
        env=os.environ.copy();env['CUDA_VISIBLE_DEVICES']=str(i)
        with open(OUT/f'view_{name}.log','w') as f:
            p=subprocess.Popen([sys.executable,'-u',__file__,'--worker',name,'--deadline',str(deadline-25)],env=env,stdout=f,stderr=subprocess.STDOUT)
        jobs.append((name,p))
    while any(p.poll() is None for _,p in jobs):
        if time.time()>deadline-15:
            for _,p in jobs:
                if p.poll() is None:p.terminate()
            break
        time.sleep(4)
    for name,p in jobs:
        try:p.wait(timeout=4)
        except subprocess.TimeoutExpired:p.kill();p.wait()
        print(name,'return code',p.returncode,flush=True);print((OUT/f'view_{name}.log').read_text()[-1600:],flush=True)
    combine(data)

if __name__=='__main__':
    p=argparse.ArgumentParser();p.add_argument('--gpu',action='store_true');p.add_argument('--deadline',type=float);p.add_argument('--worker',choices=list(VIEWS));a=p.parse_args(); deadline=a.deadline or time.time()+14*60
    if a.worker:worker(a.worker,deadline)
    else:main(a.gpu,deadline)


In [ ]:
%%writefile /kaggle/working/amml_repro_cache/solution_rank_fusion.py
"""AMML: CPU-only exploratory rank fusion. Reuses fine-tuned model outputs.
No test labels, external training data or additional neural training.
The holdouts were previously reused. Cross-fit is for recipe selection only.
"""
import os, json, importlib.util
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from IPython.display import display
ROOT=Path(os.environ.get('AMML_INPUT','/kaggle/input'))
OUT=Path(os.environ.get('AMML_OUTPUT','/kaggle/working'))
OUT.mkdir(parents=True,exist_ok=True)
SEED=42
B=20000

def one(name):
    paths=sorted(ROOT.rglob(name))
    assert len(paths)==1, f'Expected exactly one {name}: {paths}'
    return paths[0]

def previous():
    spec=importlib.util.spec_from_file_location('previous_blend',one('solution_final_blend.py'))
    m=importlib.util.module_from_spec(spec);spec.loader.exec_module(m)
    return m

def rank_columns(m,df,X,rrf=False):
    Y=np.empty_like(X,dtype=float)
    for g in m.groups(df):
        for j in range(X.shape[1]):
            r=pd.Series(X[g,j]).rank(method='average',ascending=False).to_numpy()
            Y[g,j]=1/(5+r) if rrf else -r
    return np.column_stack([m.z(df,Y[:,j]) for j in range(Y.shape[1])])

def agreement_weights(m,df,X):
    # Each query gets one weight. Average pairwise Spearman correlation,
    # using average ranks for ties; constant model columns contribute zero.
    # Fixed conservative mapping: 0.35 to 0.55, no learned test-dependent weights.
    o=np.empty(len(df),dtype=float)
    for g in m.groups(df):
        R=pd.DataFrame(X[g]).rank(axis=0,method='average').to_numpy()
        R=R-R.mean(axis=0); norm=np.linalg.norm(R,axis=0)
        R=R/np.maximum(norm,1e-12)
        C=R.T@R; a=C[np.triu_indices(X.shape[1],1)].mean()
        o[g]=.35+.20*np.clip(a,0,1)
    return o

def stats(d,ix):
    d=np.asarray(d,float);lo,hi=np.quantile(d[ix].mean(1),[.025,.975])
    return dict(delta=float(d.mean()),ci_lower=float(lo),ci_upper=float(hi),
        paired_se=float(d.std(ddof=1)/np.sqrt(len(d))),wins=int((d>1e-10).sum()),
        ties=int((abs(d)<=1e-10).sum()),losses=int((d< -1e-10).sum()))

def criterion(P,rows):
    d=P[rows]-P[rows,:1]
    a=d.mean(0)-d.std(0,ddof=1)/np.sqrt(len(rows))-.0005
    a[0]=0.;return a

def main():
    m=previous()
    tr,te,sample,v,common,excl,qt,anchor,av,family,sv,st=m.inputs()
    assert set(family)=={'old','a','b'}
    family=dict(family)
    for tag in ('body','long'):
        with np.load(one(f'view_{tag}.npz'),allow_pickle=False) as f:
            a,b=f['valid'],f['test']
            assert a.shape==(len(v),) and b.shape==(len(te),)
            assert np.isfinite(a).all() and np.isfinite(b).all()
            family[tag]=(m.z(v,a),m.z(te,b))
    names=['old','a','b','body','long']
    V=np.column_stack([family[k][0] for k in names]);T=np.column_stack([family[k][1] for k in names])
    vc=v.loc[common].reset_index(drop=True);ve=v.loc[excl].reset_index(drop=True)
    assert vc.query_id.nunique()==79 and ve.query_id.nunique()==378
    qv=m.z(vc,m.table(one('v6_validation.csv')).loc[vc.id,'prediction'].to_numpy(float))
    np.testing.assert_allclose(av,.5*qv+.5*sv[common],atol=1e-8)
    jv,jt=V.mean(1),T.mean(1)
    base_v,base_t=.5*qv+.5*jv[common],.5*qt+.5*jt
    saved=m.align(m.table(one('submission_final_equal.csv')),te)
    np.testing.assert_allclose(base_t,saved,rtol=1e-6,atol=1e-7,err_msg='Attach Final_blend_GPU Version 2 only')
    print('Reference reconstruction OK; max abs difference:',float(np.max(abs(base_t-saved))))
    rv,rt=rank_columns(m,v,V).mean(1),rank_columns(m,te,T).mean(1)
    fv,ft=rank_columns(m,v,V,True).mean(1),rank_columns(m,te,T,True).mean(1)
    tv,tt=np.sort(V,axis=1)[:,1:-1].mean(1),np.sort(T,axis=1)[:,1:-1].mean(1)
    wv,wt=agreement_weights(m,v,V),agreement_weights(m,te,T)
    # On Jina-only queries adaptive uses the same Jina scores, so its effect
    # cannot be independently assessed there. It is diagnostic, not eligible
    # for automatic selection from the family-only cross-fit.
    variants={
        'equal5':(jv,base_v,base_t),
        'rank_mean':(rv,.5*qv+.5*rv[common],.5*qt+.5*rt),
        'rrf':(fv,.5*qv+.5*fv[common],.5*qt+.5*ft),
        'trimmed_mean':(tv,.5*qv+.5*tv[common],.5*qt+.5*tt),
        'adaptive':(jv,(1-wv[common])*qv+wv[common]*jv[common],(1-wt)*qt+wt*jt),
    }
    keys=list(variants)
    P=np.column_stack([m.per_query(ve,x[0][excl]) for x in variants.values()])
    C=np.column_stack([m.per_query(vc,x[1]) for x in variants.values()])
    rng=np.random.default_rng(SEED);n=len(P);nc=len(C)
    ie=rng.integers(n,size=(B,n));ic=rng.integers(nc,size=(B,nc))
    records=[]
    for i,key in enumerate(keys):
        j=stats(P[:,i]-P[:,0],ie);c=stats(C[:,i]-C[:,0],ic)
        records.append(dict(variant=key,jina_ndcg=float(P[:,i].mean()),common_ndcg=float(C[:,i].mean()),
            **{'jina_'+k:x for k,x in j.items()},**{'common_'+k:x for k,x in c.items()}))
        if i:m.save(te,sample,variants[key][2],f'submission_fusion_{key}.csv')
    result=pd.DataFrame(records);result.to_csv(OUT/'fusion_comparison.csv',index=False)
    display(result[['variant','jina_ndcg','jina_delta','jina_ci_lower','jina_ci_upper','common_ndcg','common_delta','common_ci_lower','common_ci_upper']].round(6))
    # Cross-fit only the choice among the three new family recipes + reference.
    eligible=P[:,:4];cf=np.empty(n);folds=[]
    for fold,(a,b) in enumerate(KFold(5,shuffle=True,random_state=SEED).split(P),1):
        k=int(np.argmax(criterion(eligible,a)));cf[b]=P[b,k]
        folds.append(dict(fold=fold,recipe=keys[k],n=len(b),delta=float((cf[b]-P[b,0]).mean())))
    pd.DataFrame(folds).to_csv(OUT/'fusion_folds.csv',index=False)
    proposed=int(np.argmax(criterion(eligible,np.arange(n))))
    cv=stats(cf-P[:,0],ie);shared=stats(C[:,proposed]-C[:,0],ic)
    # Conservative predeclared gate; same holdouts reused, not independent.
    accepted=bool(proposed!=0 and cv['ci_lower']>0 and shared['delta']>0 and shared['ci_lower']>=-.005)
    winner=keys[proposed] if accepted else 'equal5'
    m.save(te,sample,saved,'submission_reference_0961307.csv')
    m.save(te,sample,variants[keys[proposed]][2] if proposed else saved,'submission_fusion_candidate.csv')
    m.save(te,sample,variants[winner][2] if accepted else saved,'submission.csv')
    info=dict(reference='equal5',proposed=keys[proposed],selected=winner,accepted=accepted,
        crossfit=cv,common_candidate=shared,common_queries=nc,jina_only_queries=n,
        adaptive_test_weight_min=float(wt.min()),adaptive_test_weight_max=float(wt.max()),
        adaptive_test_weight_mean_by_query=float(np.mean([wt[g[0]] for g in m.groups(te)])),
        adaptive_eligible_for_selection=False,bootstrap_replicates=B,
        note='Exploratory reused holdouts, multiple unadjusted CIs; no Public score used in recipe tuning. Adaptive only evaluated for full ensemble on common queries.')
    (OUT/'fusion_metrics.json').write_text(json.dumps(info,indent=2),encoding='utf-8');print(json.dumps(info,indent=2))
    import matplotlib.pyplot as plt
    fig,ax=plt.subplots(figsize=(9,4.5))
    for i,row in result.iloc[1:].reset_index(drop=True).iterrows():
        ax.plot([row.common_ci_lower,row.common_ci_upper],[i,i],lw=2)
        ax.scatter(row.common_delta,i,s=30)
    ax.axvline(0,ls='--',color='black',lw=1);ax.set_yticks(range(4),keys[1:]);ax.invert_yaxis()
    ax.set_xlabel('NDCG@10 difference relative to equal5');ax.set_title('79 common queries: exploratory paired bootstrap (95% CI)');ax.grid(axis='x',alpha=.2)
    fig.tight_layout();fig.savefig(OUT/'fusion_confidence_intervals.png',dpi=160);plt.show();plt.close(fig)

if __name__=='__main__':main()


## Запустить выбранную стадию

Финальный результат сохраняется **только** на `STAGE="final"`. Повторный запуск на другом GPU может дать небольшие отличия из-за вычислений с плавающей точкой.

In [ ]:
# Execute exactly one final-pipeline stage; earlier stages are restored from
# your single previous Kaggle Notebook Output, not downloaded from submissions.
import numpy as np, pandas as pd

def module_from_file(basename):
    spec=importlib.util.spec_from_file_location('stage_'+basename.replace('.py',''),CACHE/basename)
    mod=importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

def check_gpu():
    import torch
    assert torch.cuda.is_available(), 'Enable Kaggle GPU (T4 x2) for this stage'
    print('GPU:',torch.cuda.get_device_name(0), 'torch:',torch.__version__)

def require(path):
    assert (CACHE/path).exists(), f'Missing stage prerequisite: {path}'

if STAGE == 'check_data':
    print('Competition CSV check passed.')

elif STAGE in ('qwen4b','qwen8b'):
    check_gpu()
    tag='4b' if STAGE=='qwen4b' else '8b'
    if tag=='8b':require('v5_20261116_064730_4b/run.json')
    count=7733 if tag=='4b' else 7808
    env=os.environ.copy();env['CUDA_VISIBLE_DEVICES']='0'
    subprocess.run([sys.executable,str(CACHE/'solution_v5_worker.py'), '--worker',tag,
        '--hours','11.4','--run-id','20261116_064730','--fixed-batches',str(count)],
        env=env,check=True)
    path=CACHE/f'v5_20261116_064730_{tag}/run.json'
    d=json.loads(path.read_text())
    assert d['completed_microbatches']==[count],d
    assert (path.parent/'adapter/adapter_model.safetensors').exists()
    print('Trained',tag,'successful optimizer updates:',d['successful_updates'])

elif STAGE in ('qwen_v6_4b','qwen_v6_8b'):
    check_gpu()
    for tag in ('4b','8b'):require(f'v5_20261116_064730_{tag}/run.json')
    tag='4b' if STAGE.endswith('4b') else '8b'
    env=os.environ.copy();env['CUDA_VISIBLE_DEVICES']='0'
    # Exact v6 reference run made ZERO new gradient updates; only the extra view.
    subprocess.run([sys.executable,str(CACHE/'solution_v6_worker.py'),
        '--mode',tag,'--hours','11.4','--run-id','20261116_135727'],env=env,check=True)
    require(f'v6_20261116_135727_{tag}/view/complete.json')
    d=json.loads((CACHE/f'v6_20261116_135727_{tag}/run.json').read_text())
    assert d['completed_pairs']==0

elif STAGE=='qwen_combine':
    for tag in ('4b','8b'): require(f'v6_20261116_135727_{tag}/view/test.csv')
    mod=module_from_file('solution_v6_worker.py')
    mod.combine('20261116_135727')
    require('submission_v6_selected.csv');require('v6_validation.csv')
    info=json.loads((CACHE/'v6_metrics.json').read_text())
    assert info['selected']['weights']=={'v5':0.75,'4b_view':0.25}

elif STAGE=='jina_fast':
    check_gpu()
    mod=module_from_file('solution_fast.py')
    frame,a,v,te,sample=mod.prepare()
    mod.worker('rerank',time.time()+11.4*3600)
    require('fast_rerank_learned.safetensors')
    d=json.loads((CACHE/'fast_rerank_run.json').read_text())
    assert d['batches']==2586 and d['optimizer_updates']>0
    # QA/residual calibration is not part of the final model; do not run it.

elif STAGE in ('jina_a','jina_b'):
    check_gpu()
    require('fast_rerank_learned.safetensors')
    tag='a' if STAGE=='jina_a' else 'b'
    mod=module_from_file('solution_jina_refine.py')
    mod.prepare()
    mod.worker_refine(tag,time.time()+11.4*3600)
    require(f'jina_{tag}_learned.safetensors')
    d=json.loads((CACHE/f'jina_{tag}_run.json').read_text())
    expected=4877 if tag=='a' else 2586
    assert abs(d['best_updates']-expected)<=10, (d['best_updates'],expected)

elif STAGE=='jina_combine':
    for tag in ('a','b'):require(f'jina_{tag}_learned.safetensors')
    mod=module_from_file('solution_jina_refine.py')
    frame,a,v,te,sample=mod.prepare()
    mod.combine_refine(frame,v,te,sample)
    require('jina_refine_validation.csv');require('submission_jina_qwen_50.csv')

elif STAGE in ('view_body','view_long'):
    check_gpu()
    tag=STAGE.replace('view_','')
    require('jina_refine_validation.csv')
    mod=module_from_file('solution_final_blend.py')
    data=mod.inputs()
    mod.prepare_views(data)
    mod.worker(tag,time.time()+11.4*3600)
    require(f'view_{tag}.npz')

elif STAGE=='final':
    for name in ('view_body.npz','view_long.npz','v6_validation.csv',
                 'jina_a_validation.csv','jina_b_validation.csv'):require(name)
    m=module_from_file('solution_final_blend.py')
    ranking=module_from_file('solution_rank_fusion.py')
    tr,te,sample,v,common,excl,qtest,anchor,av,family,sv,st=m.inputs()
    assert set(family)=={'old','a','b'}
    for tag in ('body','long'):
        with np.load(CACHE/f'view_{tag}.npz',allow_pickle=False) as z:
            raw_val,raw_test=z['valid'],z['test']
        assert len(raw_val)==len(v) and len(raw_test)==len(te)
        family[tag]=(m.z(v,raw_val),m.z(te,raw_test))
    keys=('old','a','b','body','long')
    V=np.column_stack([family[k][0] for k in keys])
    T=np.column_stack([family[k][1] for k in keys])
    rv=ranking.rank_columns(m,v,V).mean(axis=1)
    rt=ranking.rank_columns(m,te,T).mean(axis=1)
    # EXACT final formula as submission_fusion_rank_mean.csv:
    # 50% query-z-scored Qwen v6 + 50% mean of five query-z-scored
    # negative Jina document ranks; all 13,507 candidates are scored.
    score=0.5*qtest+0.5*rt
    m.save(te,sample,score,'submission.csv')
    vc=v.loc[common].reset_index(drop=True)
    qv=m.z(vc,m.table(CACHE/'v6_validation.csv').loc[vc.id,'prediction'].to_numpy(float))
    local_ndcg=m.per_query(vc,0.5*qv+0.5*rv[common]).mean()
    print('Common 79-query NDCG@10:',local_ndcg)
    print('Expected original local NDCG@10 approximately 0.955617')
    out=pd.read_csv(CACHE/'submission.csv',dtype={'id':str})
    assert len(out)==13507 and out.id.is_unique and out.prediction.notna().all()
    assert set(out.id)==set(te.id) and np.isfinite(out.prediction).all()
    print('FINAL FILE:',CACHE/'submission.csv')
    print('Original selected submission: Public 0.964651, Private 0.967919 (not re-evaluated).')

# This marker is updated only if the whole selected stage finished successfully.
completed.append(STAGE)
state_file.write_text(json.dumps(completed,ensure_ascii=False,indent=2),encoding='utf-8')
print('Stage committed successfully:',STAGE, '| Completed:',completed)
